### A. Data Understanding

A0. Persiapan

Impor library serta pemeriksaan path dan DTD. Berkas XML berada di luar repo, sehingga path-nya diatur di satu variabel konfigurasi. Nama DTD tidak ditulis manual, melainkan dibaca dari header XML, karena XML sendiri yang menentukan DTD yang dipakainya. Cell ini memastikan kedua berkas ada di folder yang sama sebelum kita melanjutkan, dan berhenti dengan pesan jelas jika tidak.



In [3]:
# A0 - Imports, XML path check, DTD discovery
import hashlib, json, os, re, time
from collections import Counter
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from lxml import etree
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 150)

# The only place the XML location is set. The env var wins, so a teammate
# does not need to edit the notebook (and your personal path stays out of git).
XML_PATH = Path(os.environ.get("DBLP_XML_PATH", r"C:\Users\Mark\Documents\ITB\DataMining\TugasKelompok\dblp-2025-12-01.xml"))

if not XML_PATH.is_file():
    raise FileNotFoundError(f"XML not found: {XML_PATH}\nSet DBLP_XML_PATH or edit XML_PATH.")

# Read the DOCTYPE from the XML header; the XML decides which DTD it uses.
with open(XML_PATH, "rb") as f:
    head = f.read(4096).decode("latin-1")

m = re.search(r'<!DOCTYPE\s+\w+\s+SYSTEM\s+"([^"]+)"', head)
if m is None:
    print(head[:600])
    raise RuntimeError("No DOCTYPE ... SYSTEM found in the first 4096 bytes; adjust the regex.")

DTD_NAME = m.group(1)
if Path(DTD_NAME).name != DTD_NAME:
    raise RuntimeError(f"DTD name in the header is not a bare filename: {DTD_NAME!r}")

DTD_PATH = XML_PATH.parent / DTD_NAME
if not DTD_PATH.is_file():
    raise FileNotFoundError(f"XML asks for {DTD_NAME}, but it is not in {XML_PATH.parent}")

print("XML:", XML_PATH, f"({XML_PATH.stat().st_size:,} bytes)")
print("DTD named in header:", DTD_NAME)
print("DTD found at:", DTD_PATH)

XML: C:\Users\Mark\Documents\ITB\DataMining\TugasKelompok\dblp-2025-12-01.xml (4,939,110,596 bytes)
DTD named in header: dblp-2023-06-28.dtd
DTD found at: C:\Users\Mark\Documents\ITB\DataMining\TugasKelompok\dblp-2023-06-28.dtd


**A1. Data Details: sumber, tanggal pengambilan, dan ukuran**

Soal meminta kita menjelaskan sumber dataset, tanggal pengambilan, dan jumlah data. Setiap fakta berasal dari tempat yang berbeda, sehingga kita mencatat asal masing-masing. Tidak ada fakta yang dianggap “berasal dari XML” kecuali memang benar demikian. Nilai setiap fakta dicetak oleh cell di bawah.

| Fakta | Sumber | Jenis |
|---|---|---|
| Ukuran berkas XML, SHA-256 XML | Dihitung dari berkas | Dihitung |
| SHA-256 DTD | Dihitung dari berkas | Dihitung |
| Nama DTD | Header XML | Dibaca dari berkas |
| Rilis: Monthly Snapshot XML, Desember 2025 | Halaman DOI | Manual |
| Tanggal snapshot 2025-12-01 (hari) | Nama berkas, string DOI | Disimpulkan dari penamaan |
| Penerbit: Schloss Dagstuhl – Leibniz-Zentrum für Informatik | Halaman DOI | Manual |
| DOI snapshot | Halaman DOI | Manual |
| DOI DTD | Halaman Dagstuhl | Manual |
| Tanggal pengambilan 2026-11-16 | Catatan unduhan kita | Manual |
| Jumlah record | Proses ekstraksi (A3) | Dihitung, diisi setelah A3 |

In [ ]:
# A1 - Provenance table (file facts + manual facts)
def sha256_of(path, block=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while chunk := f.read(block):
            h.update(chunk)
    return h.hexdigest()

t0 = time.time()
xml_sha = sha256_of(XML_PATH)
dtd_sha = sha256_of(DTD_PATH)
print(f"Hashing took {time.time() - t0:.0f}s")

# A1b - Corrected GZ_PATH and md5 check
GZ_PATH = XML_PATH.with_name(XML_PATH.name + ".gz")   # same folder as the XML
SITE_HASH_ALGO = "md5"
SITE_HASH_GZ   = "bccb40fc73f6603ec153a6ff74f39eb9"

if not GZ_PATH.is_file():
    raise FileNotFoundError(f".gz not found: {GZ_PATH}")

h = hashlib.new(SITE_HASH_ALGO)
with open(GZ_PATH, "rb") as f:
    while chunk := f.read(1024 * 1024):
        h.update(chunk)

gz_hash = h.hexdigest()
gz_status = "MATCH" if gz_hash == SITE_HASH_GZ.strip().lower() else "DIFF"
print(f"{SITE_HASH_ALGO} of .gz:", gz_hash)
print("vs publisher:", gz_status)

# Snapshot date comes from the filename; the DOI string carries the same date.
DOI_SNAPSHOT = "10.4230/dblp.xml.2025-12-01"
DOI_DTD      = "10.4230/dblp.xml.dtd.2023-06-28"
date_in_name = re.search(r"\d{4}-\d{2}-\d{2}", XML_PATH.name).group(0)
date_in_doi  = re.search(r"\d{4}-\d{2}-\d{2}", DOI_SNAPSHOT).group(0)
print("Date in filename == date in DOI string:", date_in_name == date_in_doi)

to_rows = [
    ("Ukuran berkas XML (byte)", f"{XML_PATH.stat().st_size:,}", "Dihitung dari berkas", "Dihitung"),
    ("SHA-256 XML", xml_sha, "Dihitung dari berkas", "Dihitung"),
    ("SHA-256 DTD", dtd_sha, "Dihitung dari berkas", "Dihitung"),
    ("Nama DTD", DTD_NAME, "Header XML", "Dibaca dari berkas"),
    ("Rilis", "Monthly Snapshot XML Release of December 2025", "Halaman DOI", "Manual"),
    ("Tanggal snapshot (hari)", date_in_name, "Nama berkas, string DOI", "Disimpulkan dari penamaan"),
    ("Penerbit", "Schloss Dagstuhl – Leibniz-Zentrum für Informatik", "Halaman DOI", "Manual"),
    ("DOI snapshot", DOI_SNAPSHOT, "Halaman DOI", "Manual"),
    ("DOI DTD", DOI_DTD, "Halaman Dagstuhl", "Manual"),
    ("Tanggal pengambilan", "2026-11-16", "Catatan unduhan kita", "Manual"),
    ("Jumlah record", "(diisi setelah A3)", "Proses ekstraksi (A3)", "Dihitung"),
    # A1 - Extra row for the .gz check (insert into to_rows)
    ("MD5 berkas .gz", f"{gz_hash} ({gz_status} dengan nilai penerbit)","Dihitung dari berkas, dibandingkan dengan nilai di situs penerbit", "Dihitung + Manual"),
]
prov = pd.DataFrame(to_rows, columns=["fakta", "nilai", "sumber", "jenis"])
with pd.option_context("display.max_colwidth", None):
    display(prov)

Hashing took 17s
md5 of .gz: bccb40fc73f6603ec153a6ff74f39eb9
vs publisher: MATCH
Date in filename == date in DOI string: True


,fakta,nilai,sumber,jenis
0,Ukuran berkas XML (byte),"4,939,110,596",Dihitung dari berkas,Dihitung
1,SHA-256 XML,0f3dbcbd332674bd2e292395367888d1d39dec2a497d08d5c08b1d8298907d3e,Dihitung dari berkas,Dihitung
2,SHA-256 DTD,2d864498f9eef9298662df11e3874171ff24cd4a21ab5974f1369c90acdb1ef6,Dihitung dari berkas,Dihitung
3,Nama DTD,dblp-2023-06-28.dtd,Header XML,Dibaca dari berkas
4,Rilis,Monthly Snapshot XML Release of December 2025,Halaman DOI,Manual
5,Tanggal snapshot (hari),2025-12-01,"Nama berkas, string DOI",Disimpulkan dari penamaan
6,Penerbit,Schloss Dagstuhl – Leibniz-Zentrum für Informatik,Halaman DOI,Manual
7,DOI snapshot,10.4230/dblp.xml.2025-12-01,Halaman DOI,Manual
8,DOI DTD,10.4230/dblp.xml.dtd.2023-06-28,Halaman Dagstuhl,Manual
9,Tanggal pengambilan,2026-11-16,Catatan unduhan kita,Manual


**A2.1 DTD.** DTD mendefinisikan tipe record yang diizinkan di bawah root, daftar field bersama, isi inproceedings, dan atributnya. Kita mencetak empat deklarasi tersebut lalu mengurai daftar tipe record dan daftar field darinya. Jika suatu pola mencetak “not found”, regex harus disesuaikan sebelum kita melanjutkan.

In [ ]:
# A2.0 - Check the encodings instead of assuming them
print("First line of the XML:", head.splitlines()[0])      # look for encoding="..."

raw_dtd = DTD_PATH.read_bytes()
non_ascii = sum(b > 127 for b in raw_dtd)
print(f"DTD bytes: {len(raw_dtd):,} | non-ASCII bytes: {non_ascii:,}")
for enc in ("utf-8", "ascii"):
    try:
        raw_dtd.decode(enc)
        print(f"decodes as {enc}: yes")
    except UnicodeDecodeError as e:
        print(f"decodes as {enc}: no ({e.reason} at byte {e.start})")

print("="*20)
# A2.1 - Read the declarations from the DTD
dtd = DTD_PATH.read_text(encoding="latin-1")   # same encoding as the old notebook; not verified against the file
print(f"DTD: {DTD_PATH.name} | {len(dtd):,} chars | {dtd.count(chr(10)) + 1:,} lines\n")

# --- 1) What is declared at all (generic, no assumptions about specific names) ---
elem_names = re.findall(r"<!ELEMENT\s+(\S+)", dtd)
ent_decls  = re.findall(r'<!ENTITY\s+%\s+(\S+)\s+"([^"]*)"\s*>', dtd, re.S)
print(f"ELEMENT declarations: {len(elem_names)}")
print(" ", elem_names, "\n")
print(f"Parameter entities (<!ENTITY % ...>): {len(ent_decls)}")
print(" ", [n for n, _ in ent_decls], "\n")

# --- 2) The four declarations the notebook relies on (old patterns + a root pattern) ---
root = re.search(r"<!DOCTYPE\s+(\w+)", head).group(1)      # `head` comes from A0
patterns = {
    "root ELEMENT":          rf"<!ELEMENT\s+{root}\b[^>]*>",
    "field entity":          r'<!ENTITY\s+%\s+field\s+"[^"]*"\s*>',
    "inproceedings ELEMENT": r"<!ELEMENT\s+inproceedings\b[^>]*>",
    "inproceedings ATTLIST": r"<!ATTLIST\s+inproceedings\b[^>]*>",
}
found = {}
for label, pat in patterns.items():
    mm = re.search(pat, dtd, re.S)
    found[label] = mm.group(0) if mm else None
    print(f"--- {label} ---")
    print(found[label] if mm else f"(not found: {pat})", "\n")

# --- 3) Best-effort parse: expand %entity; references, then list the names inside ---
ENT = dict(ent_decls)
def expand(text, depth=5):
    for _ in range(depth):
        new = re.sub(r"%([\w.-]+);", lambda x: ENT.get(x.group(1), x.group(0)), text)
        if new == text:
            break
        text = new
    return text

def names_in(decl, own_name):
    toks = re.findall(r"[A-Za-z][\w.-]*", expand(decl))
    return sorted({t for t in toks if t not in {"ELEMENT", "PCDATA", "EMPTY", "ANY", own_name}})

if found["root ELEMENT"]:
    DTD_REC_TAGS = names_in(found["root ELEMENT"], root)
    print("Record types under the root (from DTD):", DTD_REC_TAGS)
if found["inproceedings ELEMENT"]:
    DTD_INPROC_CHILDREN = names_in(found["inproceedings ELEMENT"], "inproceedings")
    print("inproceedings children (from DTD):", DTD_INPROC_CHILDREN)


# A2.1c - Do the declarations say anything about data types? Inline markup? Entities?
for name in ["title", "booktitle", "author", "year", "crossref"]:
    mm = re.search(rf"<!ELEMENT\s+{name}\b[^>]*>", dtd, re.S)
    print(mm.group(0) if mm else f"(not found: {name})")

mm = re.search(r'<!ENTITY\s+%\s+titlecontents\s+"[^"]*"\s*>', dtd, re.S)
print(mm.group(0) if mm else "(not found: titlecontents)")

# the full ATTLIST text with escapes visible: is the '...' line really in the file?
print(repr(found["inproceedings ATTLIST"]))

# general (non-%) entities, the kind referenced as &name; in the XML text
gen = re.findall(r'<!ENTITY\s+(?!%)(\S+)\s+"([^"]*)"\s*>', dtd)
print(len(gen), "general entities; first 5:", gen[:5])

First line of the XML: <?xml version="1.0" encoding="ISO-8859-1"?>
DTD bytes: 13,134 | non-ASCII bytes: 0
decodes as utf-8: yes
decodes as ascii: yes
DTD: dblp-2023-06-28.dtd | 13,134 chars | 369 lines

ELEMENT declarations: 41
  ['dblp', 'article', 'inproceedings', 'proceedings', 'book', 'incollection', 'phdthesis', 'mastersthesis', 'www', 'data', 'person', 'author', 'editor', 'address', 'title', 'booktitle', 'pages', 'year', 'journal', 'volume', 'number', 'month', 'url', 'ee', 'cite', 'school', 'publisher', 'note', 'cdrom', 'crossref', 'isbn', 'chapter', 'rel', 'stream', 'series', 'publnr', 'ref', 'sup', 'sub', 'i', 'tt'] 

Parameter entities (<!ENTITY % ...>): 2
  ['field', 'titlecontents'] 

--- root ELEMENT ---
<!ELEMENT dblp (article|inproceedings|proceedings|book|incollection|
                phdthesis|mastersthesis|www|person|data)*> 

--- field entity ---
<!ENTITY % field "author|editor|title|booktitle|pages|year|address|journal|volume|number|month|url|ee|cdrom|cite|publisher|

**A2.2 Profil empiris.** Satu kali pass streaming pada XML, hanya melihat anak langsung dari root (sehingga tidak ada daftar tag yang diasumsikan). Untuk setiap tipe record, kita mengukur atribut yang muncul, elemen anak yang muncul, berapa record yang memuat masing-masing, dan jumlah maksimum dalam satu record. Nilai maksimum inilah yang membedakan “list” dan “scalar”: author yang berulang berarti list, sedangkan year yang tidak pernah berulang berarti scalar. Kita juga menghitung tag yang bersarang di dalam field (yang menjelaskan mengapa ekstraksi teks harus menangani markup inline) dan jumlah bagian yang dipisahkan / pada setiap key (dasar dari venue_key).

Biaya: pass ini lambat. Gunakan PROFILE_LIMIT untuk melihat sekilas terlebih dahulu. Namun hasil yang dibatasi bisa melewatkan field yang jarang muncul, karena berkas tidak berurutan acak, jadi hanya laporkan angka dari run penuh.

In [ ]:
# A2.2 - Streaming profile of the XML (tags, attributes, children, nesting, key shape)
PROFILE_LIMIT = None   # quick peek only; set to None for the full run (the only run whose numbers we report)

type_n     = Counter()    # record type -> records seen
attr_n     = Counter()    # (type, attribute) -> records carrying it
child_n    = Counter()    # (type, child tag) -> records containing it at least once
child_max  = Counter()    # (type, child tag) -> max occurrences inside one record
nested_n   = Counter()    # (type, child tag, nested tag) -> occurrences
key_parts  = Counter()    # (type, number of '/' parts in key) -> records
publtype_v = Counter()    # (type, publtype value) -> records

t0, seen = time.time(), 0
ctx = etree.iterparse(str(XML_PATH), events=("end",),
                      load_dtd=True, resolve_entities=True, huge_tree=True)
for _, el in ctx:
    parent = el.getparent()
    if parent is None or parent.getparent() is not None:
        continue                                   # only direct children of the root
    rtype = el.tag
    type_n[rtype] += 1
    for a in el.attrib:
        attr_n[(rtype, a)] += 1
    pt = el.get("publtype")
    if pt is not None:
        publtype_v[(rtype, pt)] += 1
    k = el.get("key")
    key_parts[(rtype, len(k.split("/")) if k else 0)] += 1

    kids = [c for c in el if isinstance(c.tag, str)]   # skip comments / processing instructions
    for tag, n in Counter(c.tag for c in kids).items():
        child_n[(rtype, tag)] += 1
        if n > child_max[(rtype, tag)]:
            child_max[(rtype, tag)] = n
    for c in kids:
        for d in c.iterdescendants():
            if isinstance(d.tag, str):
                nested_n[(rtype, c.tag, d.tag)] += 1

    el.clear()
    while el.getprevious() is not None:
        del parent[0]
    seen += 1
    if seen % 1_000_000 == 0:
        print(f"{seen:,} records, {time.time() - t0:.0f}s")
    if PROFILE_LIMIT and seen >= PROFILE_LIMIT:
        break

mode = f"LIMITED RUN ({PROFILE_LIMIT:,}), SLICE ONLY: layout check, not for the report" if PROFILE_LIMIT else "full run"
print(f"Records seen: {seen:,} in {time.time() - t0:.0f}s | {mode}\n")
display(pd.Series(type_n).sort_values(ascending=False).rename("records").to_frame())

to_rows = [{"child": tag, "records_with": n,
         "pct_records": round(100 * n / type_n["inproceedings"], 2),
         "max_per_record": child_max[(t, tag)]}
        for (t, tag), n in child_n.items() if t == "inproceedings"]
if to_rows:
    inproc = pd.DataFrame(to_rows).sort_values("records_with", ascending=False)
    inproc["shape"] = inproc["max_per_record"].map(lambda m: "list" if m > 1 else "scalar")
    print("inproceedings children:")
    display(inproc.reset_index(drop=True))
    print("inproceedings attributes:", {a: n for (t, a), n in attr_n.items() if t == "inproceedings"})
    print("nested tags inside inproceedings fields:",
          {(c, d): n for (t, c, d), n in nested_n.items() if t == "inproceedings"})
    print("publtype values (inproceedings):",
          {v: n for (t, v), n in publtype_v.items() if t == "inproceedings"})
else:
    print("No inproceedings in this slice: a limited run can miss whole record types.")

print("key parts per record type (number of '/' separated parts -> records):")
display(pd.Series(key_parts).unstack(fill_value=0))

# A2.2b - Save the profile counters to JSON (run after A2.2)
OUT_DIR = Path("data")            # same folder the old notebook used for outputs
OUT_DIR.mkdir(exist_ok=True)

full_run = PROFILE_LIMIT is None
PROFILE_JSON = OUT_DIR / ("dblp_profile.json" if full_run else "dblp_profile_LIMITED.json")

def to_rows(counter):
    """Counter with tuple keys -> list of [*key, count]; JSON has no tuple keys."""
    return [list(k) + [v] for k, v in sorted(counter.items(), key=lambda kv: str(kv[0]))]

profile = {
    "xml_sha256": xml_sha,                 # from A1: ties the profile to this exact file
    "dtd": DTD_NAME,
    "profile_limit": PROFILE_LIMIT,
    "full_run": full_run,
    "records_seen": seen,
    "profiled_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "type_counts": dict(type_n),
    "attr_counts": to_rows(attr_n),           # [type, attribute, records]
    "child_counts": to_rows(child_n),         # [type, child, records_with]
    "child_max": to_rows(child_max),          # [type, child, max_per_record]
    "nested_counts": to_rows(nested_n),       # [type, child, nested_tag, occurrences]
    "key_parts": to_rows(key_parts),          # [type, n_parts, records]
    "publtype_values": to_rows(publtype_v),   # [type, value, records]
}

tmp = PROFILE_JSON.with_name(PROFILE_JSON.name + ".tmp")
tmp.write_text(json.dumps(profile, indent=1, ensure_ascii=False), encoding="utf-8")
tmp.replace(PROFILE_JSON)                  # only a finished file gets the final name
print("Saved:", PROFILE_JSON, f"({PROFILE_JSON.stat().st_size:,} bytes) | full_run={full_run}")

1,000,000 records, 59s
2,000,000 records, 118s
3,000,000 records, 176s
4,000,000 records, 231s
5,000,000 records, 253s
6,000,000 records, 276s
7,000,000 records, 299s
8,000,000 records, 328s
9,000,000 records, 390s
10,000,000 records, 453s
11,000,000 records, 515s
12,000,000 records, 577s
Records seen: 12,169,828 in 585s | full run



,records
article,4101061
www,3955829
inproceedings,3788716
phdthesis,149054
incollection,70988
proceedings,63072
book,21291
data,19790
mastersthesis,27


inproceedings children:


,child,records_with,pct_records,max_per_record,shape
0,title,3788716,100.00,1,scalar
1,year,3788716,100.00,1,scalar
2,url,3788716,100.00,1,scalar
3,booktitle,3788697,100.00,1,scalar
4,crossref,3786117,99.93,2,list
5,author,3783358,99.86,279,list
6,ee,3651558,96.38,7,list
7,pages,3582752,94.56,1,scalar
8,stream,89284,2.36,1,scalar
9,cdrom,8482,0.22,2,list


inproceedings attributes: {'mdate': 3788716, 'key': 3788716, 'publtype': 5400}
nested tags inside inproceedings fields: {('title', 'i'): 6386, ('title', 'sup'): 9080, ('title', 'sub'): 2960, ('title', 'tt'): 3}
publtype values (inproceedings): {'withdrawn': 1369, 'informal': 4030, 'informal withdrawn': 1}
key parts per record type (number of '/' separated parts -> records):


,2,3,4,5
article,15,4101046,0,0
book,1,20566,724,0
data,0,19790,0,0
incollection,0,46318,24670,0
inproceedings,1,3788663,52,0
mastersthesis,9,18,0,0
phdthesis,98,148956,0,0
proceedings,0,63069,3,0
www,2,3955816,4,7


Saved: data\dblp_profile.json (17,911 bytes) | full_run=True


**A2.3 Pemeriksaan silang dan pembacaan profil.** Kita membandingkan tag record yang ditemukan di data dengan daftar DTD (PASS, atau DIFF yang dicetak untuk kedua arah), lalu membaca tabel elemen anak inproceedings: shape, pct_records, ada tidaknya di DTD, dan apakah field tersebut ada di skema saat ini. Kita mencari empat hal: title atau year yang berulang (skema berarti salah), field yang ada di data tetapi tidak ada di DTD, field umum yang kita buang, dan bentuk key yang merusak aturan venue_key.

In [18]:
# A2.3 - DTD vs data cross-check and the inproceedings field table
prof = json.loads((Path("data") / "dblp_profile.json").read_text(encoding="utf-8"))
assert prof["full_run"], "This is the LIMITED profile; run the full A2.2 first"
assert prof["xml_sha256"] == xml_sha, "Profile was made from a different XML file"

# 1) record tags, both directions
data_tags, dtd_tags = set(prof["type_counts"]), set(DTD_REC_TAGS)
print("in data, not in DTD:", sorted(data_tags - dtd_tags) or "none")
print("in DTD, not in data:", sorted(dtd_tags - data_tags) or "none")
print("PASS" if data_tags == dtd_tags else "DIFF", "- record tags in data == record tags in DTD\n")

# 2) inproceedings field table
n_in = prof["type_counts"]["inproceedings"]
cnt = {c: n for t, c, n in prof["child_counts"] if t == "inproceedings"}
mx  = {c: n for t, c, n in prof["child_max"] if t == "inproceedings"}
SCHEMA_SOURCE_TAGS = {"author", "title", "year", "booktitle", "crossref"}  # what the old SCHEMA reads (key is an attribute)

fields = sorted(set(DTD_INPROC_CHILDREN) | set(cnt))
tbl = pd.DataFrame({
    "field": fields,
    "records_with": [cnt.get(f, 0) for f in fields],
    "pct_records": [round(100 * cnt.get(f, 0) / n_in, 2) for f in fields],
    "max_per_record": [mx.get(f, 0) for f in fields],
})
tbl["shape"] = tbl["max_per_record"].map(lambda m: "absent" if m == 0 else ("list" if m > 1 else "scalar"))
tbl["in_DTD"] = tbl["field"].isin(DTD_INPROC_CHILDREN)
tbl["in_SCHEMA"] = tbl["field"].isin(SCHEMA_SOURCE_TAGS)
display(tbl.sort_values("records_with", ascending=False).reset_index(drop=True))

print("inproceedings key parts:", {p: n for t, p, n in prof["key_parts"] if t == "inproceedings"})
print("inproceedings attributes:", {a: n for t, a, n in prof["attr_counts"] if t == "inproceedings"})

in data, not in DTD: none
in DTD, not in data: ['person']
DIFF - record tags in data == record tags in DTD



,field,records_with,pct_records,max_per_record,shape,in_DTD,in_SCHEMA
0,year,3788716,100.00,1,scalar,True,True
1,title,3788716,100.00,1,scalar,True,True
2,url,3788716,100.00,1,scalar,True,False
3,booktitle,3788697,100.00,1,scalar,True,True
4,crossref,3786117,99.93,2,list,True,True
5,author,3783358,99.86,279,list,True,True
6,ee,3651558,96.38,7,list,True,False
7,pages,3582752,94.56,1,scalar,True,False
8,stream,89284,2.36,1,scalar,True,False
9,cdrom,8482,0.22,2,list,True,False


inproceedings key parts: {2: 1, 3: 3788663, 4: 52}
inproceedings attributes: {'key': 3788716, 'mdate': 3788716, 'publtype': 5400}


**A2.4 Tipe record beserta deskripsinya.** Tabel tipe record dengan satu baris deskripsi dan jumlahnya. Deskripsi diambil dari halaman FAQ dblp (https://dblp.org/faq/16154937.html), ditulis dengan membaca halaman tersebut, bukan dari ingatan. Jumlah diambil dari profil.

Catatan bacaan dari halaman itu: tipe record dblp terinspirasi BibTeX dan sedikit berbeda dari kategori publikasi di situs web dblp, sehingga kita mendeskripsikan tipe XML, bukan kategori di situs.

In [19]:
# A2.4 - Record types: description + count
DESC = {  # Indonesian paraphrase of the dblp FAQ wording; None = not read yet
    "article":       "Artikel dari jurnal atau majalah.",
    "inproceedings": "Makalah dalam proceedings konferensi atau workshop.",
    "proceedings":   "Volume proceedings sebuah konferensi atau workshop.",
    "book":          "Monograf karya penulis, atau kumpulan artikel yang disunting.",
    "incollection":  "Bagian atau bab dalam sebuah monograf.",
    "phdthesis":     "Disertasi doktor (PhD).",
    "mastersthesis": "Tesis magister; menurut FAQ jumlahnya sangat sedikit.",
    "www":           "Halaman web; menurut FAQ hanya sedikit (lihat catatan record person).",
    "data":          None,   # fill in from the FAQ page
    "person":        None,   # fill in from the FAQ page
}
# A2.4 - Replace the two placeholder descriptions
DESC["data"]   = "Ada di DTD, tetapi tidak dijelaskan di halaman FAQ yang kami baca."
DESC["person"] = "Ada di DTD; FAQ hanya menyebut bahwa record person dijelaskan di halaman terpisah. Tidak ada record bertipe ini di data."

counts = prof["type_counts"]                       # `prof` comes from A2.3
total = sum(counts.values())
rows = [(t, DESC.get(t) or "(belum dibaca dari FAQ)", n, round(100 * n / total, 2))
        for t, n in sorted(counts.items(), key=lambda kv: -kv[1])]
rows += [(t, DESC.get(t) or "(belum dibaca dari FAQ)", 0, 0.0) for t in DESC if t not in counts]
types_tbl = pd.DataFrame(rows, columns=["tipe", "deskripsi", "jumlah_record", "persen"])
types_tbl.loc[len(types_tbl)] = ["TOTAL", "", total, 100.0]
with pd.option_context("display.max_colwidth", None):
    display(types_tbl)
print("Descriptions still missing:", [t for t, d in DESC.items() if d is None])

,tipe,deskripsi,jumlah_record,persen
0,article,Artikel dari jurnal atau majalah.,4101061,33.70
1,www,Halaman web; menurut FAQ hanya sedikit (lihat catatan record person).,3955829,32.51
2,inproceedings,Makalah dalam proceedings konferensi atau workshop.,3788716,31.13
3,phdthesis,Disertasi doktor (PhD).,149054,1.22
4,incollection,Bagian atau bab dalam sebuah monograf.,70988,0.58
5,proceedings,Volume proceedings sebuah konferensi atau workshop.,63072,0.52
6,book,"Monograf karya penulis, atau kumpulan artikel yang disunting.",21291,0.17
7,data,"Ada di DTD, tetapi tidak dijelaskan di halaman FAQ yang kami baca.",19790,0.16
8,mastersthesis,Tesis magister; menurut FAQ jumlahnya sangat sedikit.,27,0.00
9,person,Ada di DTD; FAQ hanya menyebut bahwa record person dijelaskan di halaman terpisah. Tidak ada record bertipe ini di data.,0,0.00


Descriptions still missing: []


**A2.5 Keputusan skema.** Di sinilah SCHEMA dan REC_TAGS ditentukan, berdasarkan A2.1 sampai A2.4.


|Field | Bentuk (dari profil) | Dipakai? |	Alasan |
|---|---|---|---|
|author	| list |	ya	| dibutuhkan untuk hitungan author |
|title	| scalar |	ya |	pemeriksaan duplikat |
|year	| scalar	| ya	| filter, tren |
|booktitle |	scalar |	ya	| nama venue |
|key	| atribut	| ya	| ID paper, sumber venue_key |
|crossref |	scalar |	ya |	tautan ke record proceedings |
|lainnya (pages, ee, url, ...)	| sesuai profil |	tidak	| tidak dipakai task mana pun |

(Kolom “Bentuk” dan “Dipakai?” diisi dari profil yang sebenarnya, bukan dari draf ini.)

Titik keputusan sebelum lanjut: ekstraksi sudah pernah dijalankan. Jika tinjauan ini memperlihatkan kolom yang kita butuhkan nanti (misalnya untuk pemeriksaan tahun), mengubah skema berarti menghapus output dan menjalankan ulang ekstraksi. Putuskan di sini, sebelum A3.

In [20]:
# A2.5 - Schema decision table (old extraction kept; filled from the A2.3 profile)
SCHEMA_SOURCE_TAGS = {"author", "title", "year", "booktitle", "crossref"}   # + key attribute; copied from the old SCHEMA
REASON = {
    "author":    "dibutuhkan untuk hitungan author unik (Task 1, 2)",
    "title":     "pemeriksaan duplikat",
    "year":      "filter periode, tren",
    "booktitle": "nama tampilan venue",
    "crossref":  "tautan ke record proceedings (hanya nilai pertama yang disimpan)",
}
mx = {c: n for t, c, n in prof["child_max"] if t == "inproceedings"}
rows = []
for f in ["author", "title", "year", "booktitle", "crossref"]:
    rows.append((f, "list" if mx.get(f, 0) > 1 else "scalar", mx.get(f, 0), "ya", REASON[f]))
rows.append(("key (atribut)", "atribut", 1, "ya", "ID paper (paper_key), sumber venue_key; #REQUIRED di DTD"))
rows.append(("publtype (atribut)", "atribut opsional", 1, "tidak", "tidak diekstrak; tersedia pada 5.400 record inproceedings (semua tahun)"))
rest = sorted(set(mx) - SCHEMA_SOURCE_TAGS)
rows.append((", ".join(rest), "sesuai profil", max(mx[f] for f in rest), "tidak", "tidak dipakai task mana pun"))
sch = pd.DataFrame(rows, columns=["field", "bentuk", "max_per_record", "dipakai", "alasan"])
with pd.option_context("display.max_colwidth", None):
    display(sch)
print("Schema unchanged: re-extraction not needed (decision: keep the existing parquet).")

,field,bentuk,max_per_record,dipakai,alasan
0,author,list,279,ya,"dibutuhkan untuk hitungan author unik (Task 1, 2)"
1,title,scalar,1,ya,pemeriksaan duplikat
2,year,scalar,1,ya,"filter periode, tren"
3,booktitle,scalar,1,ya,nama tampilan venue
4,crossref,list,2,ya,tautan ke record proceedings (hanya nilai pertama yang disimpan)
5,key (atribut),atribut,1,ya,"ID paper (paper_key), sumber venue_key; #REQUIRED di DTD"
6,publtype (atribut),atribut opsional,1,tidak,tidak diekstrak; tersedia pada 5.400 record inproceedings (semua tahun)
7,"cdrom, cite, editor, ee, journal, month, note, number, pages, publnr, stream, url, volume",sesuai profil,137,tidak,tidak dipakai task mana pun


Schema unchanged: re-extraction not needed (decision: keep the existing parquet).


**A3. Ekstraksi data: dari XML ke tabel mentah**

Satu pass streaming melalui berkas 4,9 GB. Pass ini menghitung setiap tipe record dan menyimpan record inproceedings tahun 2015 sampai 2025 ke parquet mentah. Di sinilah ketentuan dataset dari soal diterapkan: tipe inproceedings, tahun 2015 sampai 2025. Hasil ekstraksi mentah tidak pernah diubah. Semua pembersihan selanjutnya bekerja pada salinan di memori.

Pilihan desain yang perlu disebut dalam laporan: proses menulis ke berkas .tmp terlebih dahulu, sehingga hanya berkas yang selesai yang memakai nama akhir. Proses ini melewati dirinya sendiri jika output sudah ada. Proses ini juga menulis JSON berisi jumlah dan parameter, yang inilah yang kita commit (parquet terlalu besar untuk GitHub).

In [ ]:
# A3.1 - Extraction config (old config; REC_TAGS now comes from the DTD)
OUT_DIR = Path("data")
OUT_DIR.mkdir(exist_ok=True)

YEAR_MIN, YEAR_MAX = 2015, 2025       # spec: 2015-2025
CHUNK_ROWS     = 100_000              # rows buffered in RAM before writing to parquet
MAX_RECORDS    = None                 # smoke test: e.g. 20_000 (writes *_test files); None = full pass
RUN_EXTRACTION = False                # the shared parquet already exists; True only to regenerate


# A3.1 - Versioned output names (replaces the SUFFIX lines; existing files keep their names)
EXTRACT_TAG = ""          # "" = version 1 (the shared files); "_v2" for a new extraction
TAG = "_test" if MAX_RECORDS else EXTRACT_TAG
RAW_PARQUET = OUT_DIR / f"raw_inproceedings{TAG}.parquet"
COUNTS_JSON = OUT_DIR / f"raw_counts{TAG}.json"

REC_TAGS = set(DTD_REC_TAGS)          # from A2.1

SCHEMA = pa.schema([
    ("paper_key",  pa.string()),
    ("authors",    pa.list_(pa.string())),
    ("title",      pa.string()),
    ("year_raw",   pa.string()),
    ("booktitle",  pa.string()),
    ("crossref",   pa.string()),      # first <crossref> only; the data has up to 2 (see A2.5)
    ("key_prefix", pa.string()),
    ("venue_key",  pa.string()),
])

In [24]:
# A3.2 - Parse helper and streaming extraction pass (old code; gated by RUN_EXTRACTION)
def text_of(el):
    """All text inside an element (handles inline tags like <i>), or None."""
    return "".join(el.itertext()).strip() if el is not None else None

def parse_inproceedings(el):
    key = el.get("key")
    parts = key.split("/") if key else []
    return {
        "paper_key":  key,
        "authors":    [text_of(a) for a in el.findall("author")],
        "title":      text_of(el.find("title")),
        "year_raw":   text_of(el.find("year")),
        "booktitle":  text_of(el.find("booktitle")),
        "crossref":   text_of(el.find("crossref")),
        "key_prefix": parts[0] if parts else None,                 # e.g. conf
        "venue_key":  parts[1] if len(parts) >= 3 else None,       # e.g. kdd
    }

def run_extraction():
    type_counts, year_counts, prefix_counts = Counter(), Counter(), Counter()
    total = kept = bad_year = 0
    buffer = []
    t0 = time.time()

    tmp = RAW_PARQUET.with_name(RAW_PARQUET.name + ".tmp")
    writer = pq.ParquetWriter(tmp, SCHEMA)

    def flush():
        nonlocal buffer
        if buffer:
            writer.write_table(pa.Table.from_pylist(buffer, schema=SCHEMA))
            buffer = []

    try:
        ctx = etree.iterparse(str(XML_PATH), events=("end",),
                              load_dtd=True, resolve_entities=True, huge_tree=True)
        for _, el in ctx:
            if el.tag not in REC_TAGS:
                continue
            total += 1
            type_counts[el.tag] += 1

            if el.tag == "inproceedings":
                rec = parse_inproceedings(el)
                prefix_counts[str(rec["key_prefix"])] += 1
                y = rec["year_raw"]
                if y and y.isdigit():
                    year_counts[y] += 1                  # all years, pre-filter
                    if YEAR_MIN <= int(y) <= YEAR_MAX:
                        buffer.append(rec)
                        kept += 1
                        if len(buffer) >= CHUNK_ROWS:
                            flush()
                else:
                    bad_year += 1

            el.clear()
            while el.getprevious() is not None:
                del el.getparent()[0]

            if total % 500_000 == 0:
                print(f"{total:,} records, {kept:,} kept, {time.time()-t0:.0f}s")
            if MAX_RECORDS and total >= MAX_RECORDS:
                break
        flush()
    finally:
        writer.close()

    tmp.replace(RAW_PARQUET)   # only a finished file gets the final name

    meta = {
        "snapshot": XML_PATH.stem,                    # changed: from the file name, not typed
        "dtd": DTD_PATH.name,
        "xml_size_bytes": XML_PATH.stat().st_size,
        "xml_sha256": xml_sha,                        # added: from A1
        "year_filter": [YEAR_MIN, YEAR_MAX],
        "columns": SCHEMA.names,                      # added
        "extracted_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        "elapsed_seconds": round(time.time() - t0, 1),
        "total_records": total,
        "record_type_counts": dict(type_counts),
        "inproceedings_all_years_by_year": dict(sorted(year_counts.items())),
        "inproceedings_by_key_prefix": dict(prefix_counts),
        "inproceedings_missing_or_invalid_year": bad_year,
        "rows_kept": kept,
    }
    COUNTS_JSON.write_text(json.dumps(meta, indent=2), encoding="utf-8")
    print(f"Done: {total:,} records, {kept:,} rows kept, {meta['elapsed_seconds']}s")

if not RUN_EXTRACTION:
    print("Extraction skipped (RUN_EXTRACTION = False). A3.3 checks the existing files.")
elif RAW_PARQUET.exists() or COUNTS_JSON.exists():
    raise FileExistsError(f"{RAW_PARQUET} or {COUNTS_JSON} already exists. "
                          "Delete the old output by hand first; it may be shared.")
else:
    run_extraction()

500,000 records, 215,005 kept, 38s
1,000,000 records, 471,693 kept, 80s
1,500,000 records, 724,653 kept, 119s
2,000,000 records, 957,848 kept, 158s
2,500,000 records, 1,204,584 kept, 198s
3,000,000 records, 1,443,194 kept, 237s
3,500,000 records, 1,699,733 kept, 277s
4,000,000 records, 1,898,822 kept, 318s
4,500,000 records, 1,898,822 kept, 323s
5,000,000 records, 1,898,822 kept, 329s
5,500,000 records, 1,898,822 kept, 333s
6,000,000 records, 1,898,822 kept, 338s
6,500,000 records, 1,898,822 kept, 350s
7,000,000 records, 1,898,822 kept, 359s
7,500,000 records, 1,898,822 kept, 368s
8,000,000 records, 1,899,209 kept, 379s
8,500,000 records, 1,899,209 kept, 394s
9,000,000 records, 1,899,209 kept, 409s
9,500,000 records, 1,899,209 kept, 423s
10,000,000 records, 1,899,209 kept, 438s
10,500,000 records, 1,899,209 kept, 453s
11,000,000 records, 1,899,209 kept, 468s
11,500,000 records, 1,900,342 kept, 482s
12,000,000 records, 1,901,774 kept, 497s
Done: 12,169,828 records, 1,901,774 rows kept, 

In [25]:
# A3.3 - Quick verification of the extraction (existing files; no XML pass)
for p in (RAW_PARQUET, COUNTS_JSON):
    assert p.exists(), f"missing: {p}"
meta = json.loads(COUNTS_JSON.read_text(encoding="utf-8"))
pf   = pq.ParquetFile(RAW_PARQUET)
prof = json.loads(PROFILE_JSON.read_text(encoding="utf-8"))   # from A2.2b
assert prof["full_run"], "profile is the LIMITED one; run the full A2.2 first"

checks = {
    "parquet column names == SCHEMA":              pf.schema_arrow.names == SCHEMA.names,
    "parquet rows == rows_kept (JSON)":            pf.metadata.num_rows == meta["rows_kept"],
    "JSON year filter == config":                  meta["year_filter"] == [YEAR_MIN, YEAR_MAX],
    "JSON DTD name == DTD named in XML header":    meta["dtd"] == DTD_NAME,
    "JSON XML size == current XML size":           meta["xml_size_bytes"] == XML_PATH.stat().st_size,
    "JSON record-type counts == independent profile": meta["record_type_counts"] == prof["type_counts"],
    "JSON total_records == profile records_seen":  meta["total_records"] == prof["records_seen"],
}
for name, ok in checks.items():
    print("PASS" if ok else "FAIL", "-", name)
A3_OK = all(checks.values())
print("\nA3_OK =", A3_OK)

PASS - parquet column names == SCHEMA
PASS - parquet rows == rows_kept (JSON)
PASS - JSON year filter == config
PASS - JSON DTD name == DTD named in XML header
PASS - JSON XML size == current XML size
PASS - JSON record-type counts == independent profile
PASS - JSON total_records == profile records_seen

A3_OK = True


**A4. Gambaran dataset dan pemeriksaan konsistensi**

Setelah jumlah record tersedia, kita mendeskripsikan dataset dan memeriksa apakah angka-angka yang independen saling cocok. Bagian ini menjadi sumber tabel tipe record di laporan dan paragraf “Jumlah data”.

Pemeriksaan (masing-masing mencetak PASS atau FAIL):

jumlah semua tipe record sama dengan total record
jumlah semua key prefix sama dengan jumlah inproceedings
jumlah per tahun ditambah tahun tidak valid sama dengan jumlah inproceedings
parquet sesuai dengan JSON pada jumlah baris dan jumlah per tahun
paper_key unik dan tidak ada null (ini bukti untuk “satu paper = satu key”)

Satu demonstrasi yang dibutuhkan laporan: jumlah baris parquet dibandingkan dengan total entri author. Selisihnya menunjukkan mengapa penghitungan paper harus memakai paper_key, bukan baris author-paper.

Observasi (bukan penjelasan). Distribusi seluruh tahun menunjukkan 2025 sekitar setengah dari 2024, dan ada beberapa record bertanggal 2026. Tahun 2020 juga turun dibandingkan 2019. Ini adalah observasi. Kita menandai 2025 sebagai data parsial dan tidak menyebut penyebabnya, karena belum kita selidiki

In [26]:
# A4 - Overview tables and PASS/FAIL consistency checks
raw = pd.read_parquet(RAW_PARQUET)      # stays untouched; Part B works on a copy

yrs_all = {int(y): n for y, n in meta["inproceedings_all_years_by_year"].items()}
n_total = meta["total_records"]
n_in    = meta["record_type_counts"]["inproceedings"]
n_bad   = meta["inproceedings_missing_or_invalid_year"]
n_pre   = sum(n for y, n in yrs_all.items() if y < YEAR_MIN)
n_post  = sum(n for y, n in yrs_all.items() if y > YEAR_MAX)
n_win   = sum(n for y, n in yrs_all.items() if YEAR_MIN <= y <= YEAR_MAX)

# --- 1) Extraction funnel: what the extraction kept, and what it removed ---
funnel_ext = pd.DataFrame([
    ("Semua record di XML",               n_total,                  0),
    ("Hanya tipe inproceedings",          n_in,                     n_total - n_in),
    ("Tahun valid (angka)",               n_in - n_bad,             n_bad),
    (f"Tahun >= {YEAR_MIN}",              n_in - n_bad - n_pre,     n_pre),
    (f"Tahun <= {YEAR_MAX} (hasil akhir)", n_win,                   n_post),
], columns=["langkah", "jumlah_setelah", "dibuang"])
funnel_ext["persen_dari_inproceedings"] = (funnel_ext["jumlah_setelah"] / n_in * 100).round(2)
display(funnel_ext)

# --- 2) Distribution by year (window, from parquet) and what lies outside it ---
yr = raw["year_raw"].astype(int).value_counts().sort_index()
yr_tbl = yr.rename("jumlah").to_frame()
yr_tbl["perubahan_%"] = (yr.pct_change() * 100).round(1)
display(yr_tbl)
print("Outside the window, all years: before", YEAR_MIN, "=", f"{n_pre:,}",
      "| after", YEAR_MAX, "=", {y: n for y, n in yrs_all.items() if y > YEAR_MAX})
print(f"Observation: {YEAR_MAX}/{YEAR_MAX-1} = {yr[YEAR_MAX] / yr[YEAR_MAX-1]:.3f} | "
      f"2020 vs 2019 = {(yr[2020] / yr[2019] - 1) * 100:+.1f}%   (observations, no cause claimed)")

# --- 3) Key prefixes: all years (JSON) vs the window (parquet) ---
pref_all = pd.Series(meta["inproceedings_by_key_prefix"]).rename("semua_tahun")
pref_win = raw["key_prefix"].value_counts().rename("2015-2025")
display(pd.concat([pref_all, pref_win], axis=1).fillna(0).astype("int64"))

# --- 4) Columns and a sample ---
print(raw.dtypes.to_string(), "\n")
display(raw.head())

# --- 5) PASS/FAIL checks ---
in_win_json = {y: n for y, n in yrs_all.items() if YEAR_MIN <= y <= YEAR_MAX}
checks = {
    "record types sum to total_records":
        sum(meta["record_type_counts"].values()) == n_total,
    "key prefixes sum to inproceedings (all years)":
        sum(meta["inproceedings_by_key_prefix"].values()) == n_in,
    "all-years per-year counts + invalid years == inproceedings (all years)":
        sum(yrs_all.values()) + n_bad == n_in,
    "in-window per-year counts sum to rows_kept":
        n_win == meta["rows_kept"],
    "parquet rows == rows_kept":
        len(raw) == meta["rows_kept"],
    "parquet per-year counts == JSON per-year counts (window)":
        raw["year_raw"].astype(int).value_counts().to_dict() == in_win_json,
    "parquet year_raw: digits only, inside the window":
        bool(raw["year_raw"].str.fullmatch(r"\d+").all()
             and raw["year_raw"].astype(int).between(YEAR_MIN, YEAR_MAX).all()),
    "paper_key has no nulls":
        bool(raw["paper_key"].notna().all()),
    "paper_key is unique":
        bool(raw["paper_key"].is_unique),
    "independent profile: inproceedings key parts sum to inproceedings":
        sum(n for t, p, n in prof["key_parts"] if t == "inproceedings") == n_in,
}
for name, ok in checks.items():
    print("PASS" if ok else "FAIL", "-", name)
A4_OK = all(checks.values())

# informational, not a pass/fail: do prefix counts line up with key-part counts?
pref = meta["inproceedings_by_key_prefix"]
kp = {p: n for t, p, n in prof["key_parts"] if t == "inproceedings"}
print("\nINFO - conf+journals+series =", pref.get("conf", 0) + pref.get("journals", 0) + pref.get("series", 0),
      "| keys with 3 parts =", kp.get(3))

# --- 6) Why papers are counted by paper_key, not by author-paper rows ---
n_entries = int(raw["authors"].map(len).sum())
print("\n=== Rows vs author entries ===")
print(f"parquet rows (papers):          {len(raw):,}")
print(f"total author entries:           {n_entries:,}  ({n_entries / len(raw):.2f} per paper)")
print(f"rows with an empty author list: {int(raw['authors'].map(len).eq(0).sum()):,}")
print("Counting author-paper rows would inflate the paper count by that factor.")

,langkah,jumlah_setelah,dibuang,persen_dari_inproceedings
0,Semua record di XML,12169828,0,321.21
1,Hanya tipe inproceedings,3788716,8381112,100.00
2,Tahun valid (angka),3788716,0,100.00
3,Tahun >= 2015,1901797,1886919,50.20
4,Tahun <= 2025 (hasil akhir),1901774,23,50.20


,jumlah,perubahan_%
year_raw,,
2015,161487,NaN
2016,162416,0.6
2017,168293,3.6
2018,176824,5.1
2019,188960,6.9
2020,171974,-9.0
2021,178461,3.8
2022,178510,0.0
2023,200626,12.4


Outside the window, all years: before 2015 = 1,886,919 | after 2025 = {2026: 23}
Observation: 2025/2024 = 0.512 | 2020 vs 2019 = -9.0%   (observations, no cause claimed)


,semua_tahun,2015-2025
series,57,0
conf,3774777,1898822
journals,13829,2912
www,1,0
books,51,40
persons,1,0


paper_key        str
authors       object
title            str
year_raw         str
booktitle        str
crossref         str
key_prefix       str
venue_key        str 



,paper_key,authors,title,year_raw,booktitle,crossref,key_prefix,venue_key
0,conf/dac/RoquetSRTSK24,"[Lucas Roquet, Fernando Fernandes dos Santos, ...",Cross-Layer Reliability Evaluation and Efficie...,2024,DAC,conf/dac/2024,conf,dac
1,conf/dac/LiuZWCYHYJ22,"[Fangxin Liu, Wenbo Zhao 0005, Zongwu Wang, Yo...",SATO: spiking neural network acceleration via ...,2022,DAC,conf/dac/2022,conf,dac
2,conf/dac/AhnYLHKP20,"[Sunwoo Ahn, Hayoon Yi, Younghan Lee, Whoi Ree...",Hawkware: Network Intrusion Detection based on...,2020,DAC,conf/dac/2020,conf,dac
3,conf/dac/ParkPVC24,"[Chanwoo Park, Junghwan Park, Premkumar Vincen...",Accelerating DTCO with a Sample-Efficient Acti...,2024,DAC,conf/dac/2024,conf,dac
4,conf/dac/PuL0HLC024,"[Yuan Pu 0001, Fangzhou Liu, Yu Zhang 0189, Zh...",Lesyn: Placement-aware Logic Resynthesis for N...,2024,DAC,conf/dac/2024,conf,dac


PASS - record types sum to total_records
PASS - key prefixes sum to inproceedings (all years)
PASS - all-years per-year counts + invalid years == inproceedings (all years)
PASS - in-window per-year counts sum to rows_kept
PASS - parquet rows == rows_kept
PASS - parquet per-year counts == JSON per-year counts (window)
PASS - parquet year_raw: digits only, inside the window
PASS - paper_key has no nulls
PASS - paper_key is unique
PASS - independent profile: inproceedings key parts sum to inproceedings

INFO - conf+journals+series = 3788663 | keys with 3 parts = 3788663

=== Rows vs author entries ===
parquet rows (papers):          1,901,774
total author entries:           7,432,518  (3.91 per paper)
rows with an empty author list: 1,615
Counting author-paper rows would inflate the paper count by that factor.


**A5. Ringkasan Data Understanding**

Ringkasan tertulis singkat, yang sekaligus menjadi jembatan ke Part B:

Yang sudah kita ketahui (dengan cell buktinya): struktur, tipe record, field dan bentuknya, jumlah, aturan key.
Yang belum kita ketahui: penyebab selisih 2025 dan penurunan 2020, apa sebenarnya key prefix selain conf, seberapa umum baris tanpa author, dan apakah venue_key berbasis key berlaku untuk semua key.
Dibawa ke Part B: paper_key adalah unit paper, parquet mentah tetap tidak diubah, dan filter tipe serta tahun sudah dilakukan saat ekstraksi (didokumentasikan, tidak diulang).

Setelah itu, baris jumlah record pada tabel A1 diisi. [N_final] tetap kosong sampai Part B selesai.

In [27]:
# A5 - Complete the A1 provenance table with record counts
counts_rows = [
    ("Jumlah record (semua tipe)",          f"{meta['total_records']:,}",                "Pass streaming A2.2 dan A3 (dua pass terpisah)", "Dihitung"),
    ("Jumlah inproceedings (semua tahun)",  f"{n_in:,}",                                  "Pass streaming A2.2 dan A3",                     "Dihitung"),
    (f"Jumlah inproceedings {YEAR_MIN}-{YEAR_MAX} (parquet mentah)", f"{len(raw):,}",    "Ekstraksi A3 (sesi sebelumnya), diperiksa di A3.3/A4", "Dihitung"),
    ("N_final (setelah preprocessing)",     "(diisi setelah Part B)",                    "Part B, S8",                                     "Dihitung"),
]
prov = prov[prov["fakta"] != "Jumlah record"]
prov = pd.concat([prov, pd.DataFrame(counts_rows, columns=prov.columns)], ignore_index=True)
with pd.option_context("display.max_colwidth", None):
    display(prov)

,fakta,nilai,sumber,jenis
0,Ukuran berkas XML (byte),"4,939,110,596",Dihitung dari berkas,Dihitung
1,SHA-256 XML,0f3dbcbd332674bd2e292395367888d1d39dec2a497d08d5c08b1d8298907d3e,Dihitung dari berkas,Dihitung
2,SHA-256 DTD,2d864498f9eef9298662df11e3874171ff24cd4a21ab5974f1369c90acdb1ef6,Dihitung dari berkas,Dihitung
3,Nama DTD,dblp-2023-06-28.dtd,Header XML,Dibaca dari berkas
4,Rilis,Monthly Snapshot XML Release of December 2025,Halaman DOI,Manual
5,Tanggal snapshot (hari),2025-12-01,"Nama berkas, string DOI",Disimpulkan dari penamaan
6,Penerbit,Schloss Dagstuhl – Leibniz-Zentrum für Informatik,Halaman DOI,Manual
7,DOI snapshot,10.4230/dblp.xml.2025-12-01,Halaman DOI,Manual
8,DOI DTD,10.4230/dblp.xml.dtd.2023-06-28,Halaman Dagstuhl,Manual
9,Tanggal pengambilan,2026-11-16,Catatan unduhan kita,Manual


## Task 1 — Data Understanding and Preprocessing

Part A menghasilkan parquet mentah yang **tidak pernah diubah**. Seluruh preprocessing bekerja pada salinan di memori (`df`). Setiap langkah mencatat jumlah baris sebelum dan sesudah ke `funnel`, dan setiap baris yang dibuang disimpan di `df_rejected` beserta langkah dan alasannya. Dengan begitu, setiap angka di laporan bisa ditelusuri ke sebuah cell.

Dua keputusan yang dibawa dari Part A:
- Tipe dan tahun sudah difilter saat ekstraksi, sehingga tidak diulang di sini.
- Baris dengan key prefix selain `conf` **tidak dihapus**. Baris tersebut hanya diberi flag `in_conf_prefix`, dan dampaknya diuji di Task 2.

**Cell T00 – Working copy, funnel log, rejected log**

In [40]:
# T00 - Working copy, funnel log, rejected-rows log
df = raw.copy()                                   # `raw` from A4 stays untouched
df["authors"] = df["authors"].map(list)           # parquet returns numpy arrays; plain lists are safer
df["in_conf_prefix"] = df["key_prefix"].eq("conf")  # flag only, nothing is deleted on prefix

funnel = []
def log_step(step, before, after, note=""):
    funnel.append({"step": step, "rows_before": before, "rows_after": after,
                   "removed": before - after, "note": note})

df_rejected = df.iloc[0:0].assign(step="", reason="")
def reject(rows, step, reason):
    """Move removed rows into the audit log. `reason` is a str or a Series aligned to rows."""
    global df_rejected
    df_rejected = pd.concat([df_rejected, rows.assign(step=step, reason=reason)],
                            ignore_index=True)

log_step("Raw extract: inproceedings, 2015-2025 (done at extraction, documentation only)",
         len(df), len(df))
display(pd.DataFrame(funnel))

,step,rows_before,rows_after,removed,note
0,"Raw extract: inproceedings, 2015-2025 (done at...",1901774,1901774,0,


### T01. Missing values

**Aturan.** Sebuah baris dibuang jika salah satu kondisi ini terpenuhi:
- daftar author kosong (sebagaimana hasil ekstraksi, tanpa strip atau pembersihan apa pun),
- `booktitle` kosong atau hanya spasi,
- `venue_key` kosong,
- `year_raw` kosong.

Baris dengan `title` kosong **dipertahankan** dan hanya dilaporkan.

**Alasan.** Author dibutuhkan untuk hitungan author unik (Task 1, 2) dan transaksi author-based. `booktitle` dan `venue_key` mengidentifikasi konferensi, sehingga baris tanpa keduanya tidak bisa masuk peringkat mana pun. Tahun dibutuhkan untuk tren. `title` hanya dipakai pada pemeriksaan duplikat (T02), dan baris tanpa judul tetap dihitung sebagai publikasi, sehingga membuangnya hanya menghilangkan data. Pada T02 baris tersebut cukup dikecualikan dari pencocokan judul.

**Bukti.** Cell di bawah mencetak jumlah per aturan (aturan bisa tumpang tindih, jadi jumlahnya bisa lebih besar dari total yang dibuang), jumlah yang hanya gagal pada satu aturan, jumlah gabungan, dan contoh baris. Baris yang dibuang masuk ke `df_rejected` dengan kolom `reason`.

**Pertanyaan terbuka: kasus `conf/sigir/ZhouZHH16`.** Paper biasa yang tidak memiliki author di DBLP. Kita memeriksa apakah ini kasus umum atau hanya kasus tersendiri, dengan melihat baris berauthor kosong yang judulnya **bukan** front matter: jumlahnya, sebarannya per venue dan per tahun, dan contoh acak.

**Keterbatasan.** (1) Front matter yang mencantumkan ketua konferensi sebagai author tidak terjaring aturan ini. (2) Entri author yang hanya berisi string kosong baru akan terlihat setelah normalisasi (T03), sehingga diperiksa lagi pada integrity assertions.

In [41]:
# T01 - Missing values: per-rule counts, union, examples; rejected rows go to df_rejected
assert not any(s["step"].startswith("T01") for s in funnel), "T01 already applied; rerun T00 first"
n0 = len(df)

def blank(s):
    """True where a value is missing or whitespace-only."""
    return s.fillna("").astype(str).str.strip().eq("")

rules = {
    "no author":       df["authors"].map(len).eq(0),       # as extracted, no stripping
    "blank booktitle": blank(df["booktitle"]),
    "blank venue_key": blank(df["venue_key"]),
    "blank year":      blank(df["year_raw"]),
}
flags = pd.concat(rules, axis=1)
bad = flags.any(axis=1)

# --- 1) Evidence: per-rule, exclusive, and union counts ---
only_one = flags[flags.sum(axis=1) == 1].sum()
summary = pd.DataFrame({"fails_rule": flags.sum(), "fails_only_this_rule": only_one})
display(summary)
print(f"Rows failing at least one rule (union): {int(bad.sum()):,} of {n0:,} ({bad.mean()*100:.3f}%)")
print(f"Title blank (kept, reported only):      {int(blank(df['title']).sum()):,}")

# informational: author list non-empty but every entry blank (T03 would turn these into empty lists)
blank_only = df["authors"].map(lambda L: len(L) > 0 and all(not (a or "").strip() for a in L))
print(f"INFO - rows whose author entries are all blank strings: {int(blank_only.sum()):,}")

# --- 2) Examples per rule (random, not head()) ---
for name, m in rules.items():
    k = int(m.sum())
    if k:
        print(f"\n-- examples: {name} ({k:,} rows)")
        display(df.loc[m, ["paper_key", "authors", "title", "booktitle", "year_raw"]]
                  .sample(min(5, k), random_state=42))

# --- 3) Open case: empty-author rows that are NOT front matter (is ZhouZHH16 general?) ---
FM_PAT = (r"front\s?matter|\b(?:preface|foreword)\b|\bmessage from\b|table of contents"
          r"|\bcommittee\b|\bwelcome\b")
t = df["title"].fillna("")
is_fm = t.str.contains(FM_PAT, case=False, regex=True)
empty = rules["no author"]
real_empty = empty & ~is_fm

print("\n=== Empty-author rows: front matter or real papers? ===")
print(f"empty author list:                 {int(empty.sum()):,}")
print(f"  title looks like front matter:   {int((empty & is_fm).sum()):,}")
print(f"  title does NOT look like it:     {int(real_empty.sum()):,}  "
      f"({real_empty.sum() / max(empty.sum(), 1) * 100:.1f}% of empty-author rows, "
      f"{real_empty.sum() / n0 * 100:.3f}% of all rows)")

if real_empty.any():
    top_v = df.loc[real_empty, "venue_key"].value_counts()
    print("\nConcentration: top 10 venue_key =", f"{top_v.head(10).sum() / top_v.sum() * 100:.1f}%",
          f"of these rows, spread over {len(top_v):,} venue_keys")
    display(top_v.head(10).rename("rows").to_frame())
    display(df.loc[real_empty, "year_raw"].value_counts().sort_index().rename("rows").to_frame().T)
    print("Random sample of empty-author rows with a non-front-matter title:")
    display(df.loc[real_empty, ["paper_key", "title", "booktitle", "year_raw"]]
              .sample(min(15, int(real_empty.sum())), random_state=42))
print("\nThe known case:")
display(df.loc[df["paper_key"].eq("conf/sigir/ZhouZHH16"), ["paper_key", "authors", "title", "year_raw"]])

# --- 4) Apply the rules; log and audit ---
if bad.any():
    reason = flags[bad].apply(lambda r: "; ".join(flags.columns[r.values]), axis=1)
    reject(df.loc[bad], "T01 missing values", reason)
df = df.loc[~bad].copy()
log_step("T01 Drop rows missing author/booktitle/venue_key/year", n0, len(df), "blank titles kept")

# --- 5) PASS/FAIL ---
checks = {
    "rows kept + rows rejected == rows before":    len(df) + int(bad.sum()) == n0,
    "df_rejected has one row per removed paper_key": df_rejected["paper_key"].is_unique,
    "no rejected paper_key remains in df":         not df["paper_key"].isin(df_rejected["paper_key"]).any(),
    "no rule fails in df":                         not (df["authors"].map(len).eq(0).any()
                                                        or blank(df["booktitle"]).any()
                                                        or blank(df["venue_key"]).any()
                                                        or blank(df["year_raw"]).any()),
}
for name, ok in checks.items():
    print("PASS" if ok else "FAIL", "-", name)
display(pd.DataFrame(funnel))

,fails_rule,fails_only_this_rule
no author,1615,1615
blank booktitle,0,0
blank venue_key,0,0
blank year,0,0


Rows failing at least one rule (union): 1,615 of 1,901,774 (0.085%)
Title blank (kept, reported only):      0
INFO - rows whose author entries are all blank strings: 0

-- examples: no author (1,615 rows)


,paper_key,authors,title,booktitle,year_raw
264174,conf/modellierung/X20,[],Modellierung 2020 - Komplettband.,Modellierung,2020
1880755,conf/calco/X21,[],"Front Matter, Table of Contents, Preface, Conf...",CALCO,2021
1699645,conf/fat/X18,[],Preface.,FAT,2018
1531058,conf/icdt/X15,[],"Title, Table of Contents, Preface, ICDT 2015 T...",ICDT,2015
1233084,conf/ei-sda/X15,[],Front Matter: Volume 9391.,SD&A,2015



=== Empty-author rows: front matter or real papers? ===
empty author list:                 1,615
  title looks like front matter:   795
  title does NOT look like it:     820  (50.8% of empty-author rows, 0.043% of all rows)

Concentration: top 10 venue_key = 53.3% of these rows, spread over 243 venue_keys


,rows
venue_key,
icimtech,323
icip,30
siggraph,20
pkdd,16
isoen,13
ei-iss,7
ccs,7
ei-iriacv,7
ei-sda,7


year_raw,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
rows,59,93,51,67,53,37,361,54,15,23,7


... (2 outputs left out)


The known case:


,paper_key,authors,title,year_raw
34227,conf/sigir/ZhouZHH16,[],Transfer Learning for Cross-Lingual Sentiment ...,2016


PASS - rows kept + rows rejected == rows before
PASS - df_rejected has one row per removed paper_key
PASS - no rejected paper_key remains in df
PASS - no rule fails in df


,step,rows_before,rows_after,removed,note
0,"Raw extract: inproceedings, 2015-2025 (done at...",1901774,1901774,0,
1,T01 Drop rows missing author/booktitle/venue_k...,1901774,1900159,1615,blank titles kept


### T02. Konsistensi tahun

**Aturan.** `year_raw` diubah menjadi integer (`year`). Tidak ada baris yang dibuang pada langkah ini. A4 sudah membuktikan bahwa `year_raw` hanya berisi digit dan berada di dalam 2015-2025, sehingga konversi ini aman dan diperiksa ulang lewat assertion.

**Pemeriksaan konsistensi.** Tahun dari DBLP dibandingkan dengan dua sinyal yang independen dari field `year`:
1. akhiran dua digit pada `paper_key` (contoh: `conf/dac/RoquetSRTSK24` menghasilkan 2024; huruf pembeda di belakang seperti `Xia21a` diabaikan),
2. tahun pada `crossref` ke record proceedings (contoh: `conf/dac/2024` menghasilkan 2024).

**Alasan tidak ada koreksi otomatis.** Ketidakcocokan belum tentu kesalahan. Konferensi yang diadakan pada Desember bisa diterbitkan dengan tahun berikutnya, dan akhiran key dibuat oleh editor DBLP dengan aturan yang tidak kita periksa. Karena itu `year` dari DBLP dipertahankan, dan tingkat ketidakcocokan serta contohnya dilaporkan. Sebuah tahun baru dikoreksi jika satu kelas ketidakcocokan terbukti sebagai kesalahan nyata.

**Keterbatasan.** (1) Filter tahun 2015-2025 dijalankan saat ekstraksi, sebelum pemeriksaan apa pun. Paper yang salah diberi tahun 2014 atau 2026 sudah terpotong dan tidak terlihat di sini. (2) Kedua sinyal tidak tersedia untuk semua baris. Cakupannya dilaporkan di bawah, dan angka kecocokan hanya berlaku untuk baris yang memiliki sinyal.

In [42]:
# T02 - Year: cast to int, then compare against two independent signals (key suffix, crossref)
assert not any(s["step"].startswith("T02") for s in funnel), "T02 already applied; rerun T00 and T01 first"
n0 = len(df)

# --- 1) Cast year_raw -> year (A4 proved digits-only and inside the window; re-assert here) ---
assert df["year_raw"].str.fullmatch(r"\d{4}").all(), "non-4-digit year_raw found"
df["year"] = df["year_raw"].astype("int16")

# --- 2) Signal A: 2-digit suffix of the last key segment, trailing lowercase letters ignored ---
last_seg = df["paper_key"].str.rsplit("/", n=1).str[-1]
key_suffix = last_seg.str.extract(r"(\d{2})[a-z]*$")[0]
key_year = pd.to_numeric(key_suffix) + 2000            # NaN where the key has no suffix

# --- 3) Signal B: 4-digit year at the start of the last crossref segment ---
xref_seg = df["crossref"].fillna("").str.rsplit("/", n=1).str[-1]
xref_year = pd.to_numeric(xref_seg.str.extract(r"^((?:19|20)\d{2})(?!\d)")[0])

def state(signal):
    """'agree' / 'disagree' vs df['year'], 'n/a' where the signal is missing."""
    return signal.eq(df["year"]).map({True: "agree", False: "disagree"}).where(signal.notna(), "n/a")

key_state, xref_state = state(key_year), state(xref_year)

# --- 4) Evidence: coverage and agreement per signal ---
rows = []
for name, sig, st in [("paper_key suffix", key_year, key_state), ("crossref year", xref_year, xref_state)]:
    has = int(sig.notna().sum())
    agree = int((st == "agree").sum())
    rows.append({"signal": name, "has_signal": has, "coverage_%": round(has / n0 * 100, 2),
                 "agree": agree, "disagree": has - agree,
                 "agreement_% (of rows with signal)": round(agree / has * 100, 3) if has else None})
display(pd.DataFrame(rows))

print("\nCross-tab: key suffix (rows) vs crossref year (columns)")
display(pd.crosstab(key_state.rename("key"), xref_state.rename("crossref")))

# --- 5) Shape of the mismatches: year minus signal, and where they concentrate ---
for name, sig, st in [("paper_key suffix", key_year, key_state), ("crossref year", xref_year, xref_state)]:
    d = (df["year"] - sig)[st == "disagree"]
    if len(d):
        print(f"\n-- {name}: (year - signal) over {len(d):,} disagreeing rows")
        display(d.astype(int).value_counts().head(10).rename("rows").to_frame().T)

any_dis = (key_state == "disagree") | (xref_state == "disagree")
if any_dis.any():
    print(f"\nRows disagreeing on at least one signal: {int(any_dis.sum()):,} ({any_dis.mean()*100:.3f}%)")
    print("Top venue_key among them (count, and share of that venue's own rows):")
    top = df.loc[any_dis, "venue_key"].value_counts().head(10).rename("disagree_rows").to_frame()
    top["venue_rows"] = df["venue_key"].map(df["venue_key"].value_counts()).loc[top.index] if False else \
                        df["venue_key"].value_counts().reindex(top.index).values
    top["share_%"] = (top["disagree_rows"] / top["venue_rows"] * 100).round(1)
    display(top)
    print("Random sample of disagreeing rows:")
    display(df.loc[any_dis, ["paper_key", "crossref", "year"]]
              .assign(key_year=key_year[any_dis], xref_year=xref_year[any_dis], booktitle=df.loc[any_dis, "booktitle"])
              .sample(min(15, int(any_dis.sum())), random_state=42))

# --- 6) Keep DBLP's year; store the check results as audit columns (dropped at export) ---
df["year_vs_key"], df["year_vs_xref"] = key_state, xref_state
log_step("T02 Year: cast year_raw to int16, consistency check vs key suffix and crossref (no correction)",
         n0, len(df), f"rows disagreeing on >=1 signal: {int(any_dis.sum())}")

# --- 7) PASS/FAIL ---
checks = {
    "no rows removed":                       len(df) == n0,
    "year is an integer dtype":              pd.api.types.is_integer_dtype(df["year"]),
    "year has no nulls":                     bool(df["year"].notna().all()),
    "year inside 2015-2025":                 bool(df["year"].between(YEAR_MIN, YEAR_MAX).all()),
    "year == int(year_raw) for every row":   bool((df["year"] == df["year_raw"].astype(int)).all()),
    "paper_key still unique":                bool(df["paper_key"].is_unique),
}
for name, ok in checks.items():
    print("PASS" if ok else "FAIL", "-", name)
display(pd.DataFrame(funnel))


,signal,has_signal,coverage_%,agree,disagree,agreement_% (of rows with signal)
0,paper_key suffix,1900158,100.00,1893093,7065,99.628
1,crossref year,1895365,99.75,1892527,2838,99.850



Cross-tab: key suffix (rows) vs crossref year (columns)


crossref,agree,disagree,n/a
key,,,
agree,1887947,2163,2983
disagree,4579,675,1811
n/a,1,0,0



-- paper_key suffix: (year - signal) over 7,065 disagreeing rows


,1,-1,2,-10,10,-11,-9,-28,9,-3
rows,2186,1216,944,590,236,69,65,47,41,38



-- crossref year: (year - signal) over 2,838 disagreeing rows


... (3 outputs left out)

Random sample of disagreeing rows:


,paper_key,crossref,year,key_year,xref_year,booktitle
461831,conf/icarcv/BasitHH024,conf/icarcv/2022,2024,2024.0,2022.0,ICARCV
460960,conf/icarcv/ChenL0S024,conf/icarcv/2022,2024,2024.0,2022.0,ICARCV
1901736,books/sp/22/Hess22,books/sp/22/FHKMPRRS2022,2020,2022.0,NaN,Denert Award
306695,conf/icuimc/ParkHHL14,conf/icuimc/2015,2015,2014.0,2015.0,IMCOM
1900844,journals/corr/abs-2108-10494,journals/corr/abs-2108-09624,2021,2094.0,NaN,EXPRESS/SOS
639104,conf/elinfocom/ChoiKKKI25,conf/elinfocom/2024,2025,2025.0,2024.0,ICEIC
36753,conf/sysose/BassamS25,conf/sysose/2023,2025,2025.0,2023.0,SoSE
1102269,conf/ism2/ZenisekBFP0LA24,conf/ism2/2024,2023,2024.0,2024.0,ISM
242810,conf/icer/Mackay22,conf/icer/2023-2,2023,2022.0,2023.0,ICER (2)
1899272,journals/corr/abs-1805-02175,journals/corr/abs-1901-09476,2018,2075.0,NaN,QPL


PASS - no rows removed
PASS - year is an integer dtype
PASS - year has no nulls
PASS - year inside 2015-2025
PASS - year == int(year_raw) for every row
PASS - paper_key still unique


,step,rows_before,rows_after,removed,note
0,"Raw extract: inproceedings, 2015-2025 (done at...",1901774,1901774,0,
1,T01 Drop rows missing author/booktitle/venue_k...,1901774,1900159,1615,blank titles kept
2,"T02 Year: cast year_raw to int16, consistency ...",1900159,1900159,0,rows disagreeing on >=1 signal: 9228


In [43]:
# T02b - Patch: key-suffix signal is only valid for conf/ keys; recompute the evidence
is_conf_key = df["key_prefix"].eq("conf")
df.loc[~is_conf_key, "year_vs_key"] = "n/a"

k_dis = df["year_vs_key"].eq("disagree")
x_dis = df["year_vs_xref"].eq("disagree")
both, any_dis = k_dis & x_dis, k_dis | x_dis

has_key = df["year_vs_key"].ne("n/a")
print(f"key signal: {int(has_key.sum()):,} rows, agreement {(~k_dis[has_key]).mean()*100:.3f}%")
print(f"disagree on >=1 signal: {int(any_dis.sum()):,} ({any_dis.mean()*100:.3f}%) | on both: {int(both.sum()):,}")

d = (df["year"] - key_year)[k_dis].astype(int)      # key_year comes from T02
print("\n(year - key signal), conf rows only:")
display(d.value_counts().head(8).rename("rows").to_frame().T)

if both.any():
    top = df.loc[both, "venue_key"].value_counts().head(10).rename("both_disagree").to_frame()
    top["venue_rows"] = df["venue_key"].value_counts().reindex(top.index).values
    top["share_%"] = (top["both_disagree"] / top["venue_rows"] * 100).round(1)
    display(top)
    display(df.loc[both, ["paper_key", "crossref", "year", "booktitle"]]
              .assign(key_year=key_year[both], xref_year=xref_year[both])
              .sample(min(10, int(both.sum())), random_state=42))

funnel[-1]["note"] = f"rows disagreeing on >=1 signal: {int(any_dis.sum())}; on both: {int(both.sum())}"

key signal: 1,897,208 rows, agreement 99.745%
disagree on >=1 signal: 7,006 (0.369%) | on both: 673

(year - key signal), conf rows only:


,1,-1,2,-10,10,-11,-9,-28
rows,1985,948,915,567,211,56,42,32


,both_disagree,venue_rows,share_%
venue_key,,,
ism2,521,1160,44.9
birthday,80,1779,4.5
eidwt,57,518,11.0
dagstuhl,9,35,25.7
ttc,5,33,15.2
chira,1,325,0.3


,paper_key,crossref,year,booktitle,key_year,xref_year
1101837,conf/ism2/BurgertWKSMN24,conf/ism2/2024,2023,ISM,2024.0,2024.0
1102064,conf/ism2/FerrettiM24,conf/ism2/2024,2023,ISM,2024.0,2024.0
1102049,conf/ism2/WulffSS24,conf/ism2/2024,2023,ISM,2024.0,2024.0
1102039,conf/ism2/SobieczkyDZ24,conf/ism2/2024,2023,ISM,2024.0,2024.0
1102776,conf/ism2/PereiraRPS23,conf/ism2/2023,2022,ISM,2023.0,2023.0
1651400,conf/ttc/CopeiZ21,conf/ttc/2021,2020,TTC/,2021.0,2021.0
1102490,conf/ism2/Zhong0WC24,conf/ism2/2024,2023,ISM,2024.0,2024.0
1102432,conf/ism2/BragliaPFGM24a,conf/ism2/2024,2023,ISM,2024.0,2024.0
919121,conf/eidwt/HenmiK19,conf/eidwt/2019,2020,EIDWT,2019.0,2019.0
1102716,conf/ism2/PadovanoCWP24,conf/ism2/2024,2023,ISM,2024.0,2024.0


### T03. Normalisasi nama author

**Aturan.** Setiap nama author diproses dengan tiga langkah: (1) normalisasi Unicode ke bentuk NFC, (2) semua deretan spasi (termasuk spasi non-breaking) diubah menjadi satu spasi lalu di-strip, (3) entri yang kosong setelah langkah 1 dan 2 dibuang. Hasilnya disimpan di kolom `author_id`, sedangkan `authors` (mentah) tidak diubah.

**Sufiks angka dipertahankan.** DBLP membedakan penulis dengan nama sama lewat sufiks empat digit (`Wei Wang 0001` dan `Wei Wang 0002` adalah orang yang berbeda). Membuang sufiks akan menggabungkan orang yang berbeda, sehingga jumlah author unik terlalu kecil. Cell di bawah menampilkan selisih jumlah author unik dengan dan tanpa sufiks sebagai bukti.

**Yang sengaja tidak dilakukan.** Tidak ada transliterasi aksara dan tidak ada penggabungan varian nama (huruf besar/kecil, aksen). Penggabungan semacam itu bisa menyatukan orang yang berbeda. Kelompok nama yang hanya berbeda huruf besar/kecil atau aksen dilaporkan sebagai **kandidat**, tidak digabung.

**Duplikat dalam satu paper.** Jika dua entri menjadi nama yang sama setelah normalisasi, hanya satu yang disimpan (urutan dipertahankan), karena relasi paper-author adalah himpunan dan ini memengaruhi `n_authors` serta transaksi pada Task 3.

**Keterbatasan.** Disambiguasi DBLP tidak sempurna: sebagian orang terpecah menjadi beberapa identitas dan sebagian lain tergabung. Hitungan author unik adalah perkiraan, bukan kebenaran mutlak.

In [44]:
# T03 - Author normalization: NFC, collapse whitespace, drop empty entries, keep numeric suffix
import random, unicodedata
assert not any(s["step"].startswith("T03") for s in funnel), "T03 already applied; rerun T00-T02 first"
n0 = len(df)

SUFFIX_RE = re.compile(r"\s\d{4}$")               # DBLP homonym suffix, e.g. "Wei Wang 0001"

def norm_name(s):
    """NFC + collapse all Unicode whitespace to single spaces + strip. Case and suffix are untouched."""
    return "" if s is None else " ".join(unicodedata.normalize("NFC", s).split())

# Normalize each distinct raw name once, then map the lists through the dict
uniq_raw = set(a for L in df["authors"] for a in L)
name_map = {a: norm_name(a) for a in uniq_raw}
uniq_norm = set(name_map.values()) - {""}

# --- 1) Evidence: what normalization changed ---
n_entries_raw = int(df["authors"].map(len).sum())
changed = [(a, n) for a, n in name_map.items() if a != n]
print(f"author entries (raw):             {n_entries_raw:,}")
print(f"distinct raw names:               {len(uniq_raw):,}")
print(f"distinct names after NFC/space:   {len(uniq_norm):,}")
print(f"distinct raw names that changed:  {len(changed):,}")
for a, n in random.Random(42).sample(changed, min(10, len(changed))):
    print(f"  {a!r} -> {n!r}")

n_ctrl = sum(any(unicodedata.category(c) in ("Cc", "Cf") for c in n) for n in uniq_norm)
print(f"INFO - names containing control/format characters (kept as is): {n_ctrl:,}")

# --- 2) Evidence: the numeric-suffix choice ---
with_suffix = sum(bool(SUFFIX_RE.search(n)) for n in uniq_norm)
uniq_base = {SUFFIX_RE.sub("", n) for n in uniq_norm}
print(f"\nids with a numeric suffix:        {with_suffix:,}")
print(f"unique ids (suffix kept):         {len(uniq_norm):,}")
print(f"unique names (suffix dropped):    {len(uniq_base):,}  "
      f"(dropping it would merge {len(uniq_norm) - len(uniq_base):,} ids)")

# --- 3) Candidates only, never merged: variants that differ by case or accents ---
def fold_case(n):
    return n.casefold()

def fold_accent(n):
    return "".join(c for c in unicodedata.normalize("NFKD", n.casefold()) if not unicodedata.combining(c))

def variant_groups(keyfunc):
    g = {}
    for n in uniq_norm:
        g.setdefault(keyfunc(n), []).append(n)
    return [sorted(v) for v in g.values() if len(v) > 1]

for label, fn in [("case only", fold_case), ("case + accents", fold_accent)]:
    grp = variant_groups(fn)
    print(f"\ncandidate groups ({label}): {len(grp):,}")
    for v in random.Random(42).sample(grp, min(8, len(grp))):
        print("  ", v)

# --- 4) Apply: clean each list, drop empty entries, dedupe within a paper (order kept) ---
ids_all = df["authors"].map(lambda L: [name_map[a] for a in L if name_map[a]])
n_blank_entries = n_entries_raw - int(ids_all.map(len).sum())
dup_rows = ids_all.map(lambda x: len(set(x)) < len(x))
print(f"\nblank author entries removed:             {n_blank_entries:,}")
print(f"rows with a repeated author after cleanup: {int(dup_rows.sum()):,}")
df["author_id"] = ids_all.map(lambda x: list(dict.fromkeys(x)))

# --- 5) Rows left with no author after cleaning go to the audit log ---
empty = df["author_id"].map(len).eq(0)
if empty.any():
    reject(df.loc[empty], "T03 author normalization", "no author after normalization")
df = df.loc[~empty].copy()
df["n_authors"] = df["author_id"].map(len).astype("int16")
log_step("T03 Author normalization (NFC, whitespace, drop empty, keep suffix, dedupe per paper)",
         n0, len(df), f"blank entries removed: {n_blank_entries}")

uniq_final = len(set(a for L in df["author_id"] for a in L))
print(f"\nrows removed in T03: {n0 - len(df):,} | unique author_id in df: {uniq_final:,}")
display(df["n_authors"].describe().round(2).to_frame().T)

# --- 6) PASS/FAIL ---
ids_in_df = set(a for L in df["author_id"] for a in L)
checks = {
    "rows kept + rejected == rows before":    len(df) + int(empty.sum()) == n0,
    "every author list is non-empty":         bool(df["author_id"].map(len).ge(1).all()),
    "no empty string in author_id":           "" not in ids_in_df,
    "no repeated author within a paper":      bool(df["author_id"].map(lambda x: len(set(x)) == len(x)).all()),
    "normalization is idempotent":            all(norm_name(x) == x for x in ids_in_df),
    "unique ids <= distinct raw names":       len(ids_in_df) <= len(uniq_raw),
    "paper_key still unique":                 bool(df["paper_key"].is_unique),
    "n_authors == len(author_id)":            bool((df["n_authors"] == df["author_id"].map(len)).all()),
}
for name, ok in checks.items():
    print("PASS" if ok else "FAIL", "-", name)
display(pd.DataFrame(funnel))

author entries (raw):             7,432,518
distinct raw names:               1,738,301
distinct names after NFC/space:   1,738,301
distinct raw names that changed:  0
INFO - names containing control/format characters (kept as is): 0

ids with a numeric suffix:        75,294
unique ids (suffix kept):         1,738,301
unique names (suffix dropped):    1,676,073  (dropping it would merge 62,228 ids)

candidate groups (case only): 2,905
   ['MiaoMiao Cheng', 'Miaomiao Cheng']
   ['YuXin Xie', 'Yuxin Xie']
   ['Andreas Rössler', 'Andreas Rößler']
   ['Julian Hossbach', 'Julian Hoßbach']
   ['ManMohan', 'Manmohan']
   ['DiJia Su', 'Dijia Su']
   ['JuEun Park', 'Jueun Park']
   ['TaeWon Seo', 'Taewon Seo']

candidate groups (case + accents): 7,286
   ['Christian Perez', 'Christian Pérez']
   ['Arturo Mayoral Lopez-de-Lerma', 'Arturo Mayoral López-de-Lerma']
   ['SeungYong Shin', 'Seungyong Shin']
   ['Markus Kramer', 'Markus Krämer']
   ['Hector R. Amado-Salvatierra', 'Héctor R. Amado-Salva

,count,mean,std,min,25%,50%,75%,max
n_authors,1900159.0,3.91,2.28,1.0,3.0,4.0,5.0,279.0


PASS - rows kept + rejected == rows before
PASS - every author list is non-empty
PASS - no empty string in author_id
PASS - no repeated author within a paper
PASS - normalization is idempotent
PASS - unique ids <= distinct raw names
PASS - paper_key still unique
PASS - n_authors == len(author_id)


,step,rows_before,rows_after,removed,note
0,"Raw extract: inproceedings, 2015-2025 (done at...",1901774,1901774,0,
1,T01 Drop rows missing author/booktitle/venue_k...,1901774,1900159,1615,blank titles kept
2,"T02 Year: cast year_raw to int16, consistency ...",1900159,1900159,0,rows disagreeing on >=1 signal: 7006; on both:...
3,"T03 Author normalization (NFC, whitespace, dro...",1900159,1900159,0,blank entries removed: 0


### T04a. Normalisasi nama conference/venue

**Tahap 1 — Pengukuran sebelum aturan ditetapkan.** Sebelum menetapkan aturan normalisasi dan pemilihan nama tampilan venue, kita mengukur struktur `key_prefix`, `venue_key`, `booktitle`, dan `crossref` pada data setelah T03 (`1.900.159` baris):
1. **Sebaran `key_prefix` (`conf`, `journals`, `books`):** jumlah baris, `venue_key` unik, serta contoh isi pada prefix non-`conf` (terutama `journals/corr`) untuk menilai apakah baris tersebut merepresentasikan konferensi/workshop atau pseudo-venue.
2. **Kebersihan string `booktitle` dan `venue_key`:** dampak normalisasi Unicode NFC dan perapian spasi, serta pemeriksaan karakter kontrol.
3. **Pemetaan dua arah `venue_key` ↔ `booktitle` bersih:**
   - Arah 1 (`venue_key` → banyak `booktitle`): seberapa sering satu `venue_key` memiliki beberapa ejaan atau track (misalnya konferensi utama dan workshop-nya), serta daftar ejaan terbanyak pada `venue_key` yang paling beragam.
   - Arah 2 (`booktitle` bersih → banyak `venue_key`): apakah ada nama venue bersih yang sama tetapi terpecah ke beberapa `venue_key`.
4. **Identifikasi workshop dan pola `crossref`:** menguji heuristik pola `booktitle` (`Workshop`, `@`) dan sampel acaknya untuk melihat apakah workshop berbagi `venue_key` dengan konferensi induk atau memakai `venue_key` tersendiri.

In [45]:
# T04a - Measure venue_key, booktitle, prefixes, and workshop patterns (read-only on df)
import unicodedata

n0 = len(df)

def norm_str(s):
    """NFC + collapse all Unicode whitespace to single spaces + strip."""
    return "" if s is None else " ".join(unicodedata.normalize("NFC", str(s)).split())

# --- 1) String cleanliness check on booktitle and venue_key ---
uniq_bt_raw = df["booktitle"].unique()
bt_map = {b: norm_str(b) for b in uniq_bt_raw}
bt_clean = df["booktitle"].map(bt_map)
bt_changed = [(a, b) for a, b in bt_map.items() if a != b]

uniq_vk_raw = df["venue_key"].unique()
vk_map = {v: norm_str(v) for v in uniq_vk_raw}
vk_changed = [(a, b) for a, b in vk_map.items() if a != b]

n_bt_ctrl = sum(any(unicodedata.category(c) in ("Cc", "Cf") for c in b) for b in bt_map.values())
print(f"distinct raw booktitle:             {len(uniq_bt_raw):,}")
print(f"distinct cleaned booktitle (NFC/ws):{bt_clean.nunique():,}")
print(f"raw booktitle strings changed:      {len(bt_changed):,}")
for a, b in bt_changed[:8]:
    print(f"  {a!r} -> {b!r}")
print(f"INFO - cleaned booktitles with control/format chars: {n_bt_ctrl:,}")
print(f"distinct venue_key:                 {len(uniq_vk_raw):,} (changed by NFC/ws: {len(vk_changed):,})")

# --- 2) Key prefix breakdown & inspection of non-conf rows ---
tmp = pd.DataFrame({
    "paper_key": df["paper_key"],
    "key_prefix": df["key_prefix"],
    "venue_key": df["venue_key"],
    "bt_clean": bt_clean,
    "crossref": df["crossref"],
})

print("\n=== Breakdown by key_prefix ===")
pref_tbl = (
    tmp.groupby("key_prefix", dropna=False)
       .agg(rows=("paper_key", "count"),
            unique_venue_key=("venue_key", "nunique"),
            unique_bt_clean=("bt_clean", "nunique"))
       .sort_values("rows", ascending=False)
)
display(pref_tbl)

for pref in ["journals", "books"]:
    sub = tmp[tmp["key_prefix"] == pref]
    if len(sub):
        print(f"\n-- Non-conf prefix '{pref}' ({len(sub):,} rows) top venue_keys:")
        display(sub["venue_key"].value_counts().head(10).rename("rows").to_frame())
        print(f"Top cleaned booktitles in '{pref}':")
        display(sub["bt_clean"].value_counts().head(10).rename("rows").to_frame())

# --- 3) Direction A: venue_key -> distinct cleaned booktitle ---
vk_stats = (
    tmp.groupby("venue_key")
       .agg(rows=("paper_key", "count"),
            n_bt=("bt_clean", "nunique"),
            in_conf=("key_prefix", lambda s: bool((s == "conf").all())))
       .sort_values("n_bt", ascending=False)
)
print("\n=== Direction A: distinct cleaned booktitles per venue_key ===")
print(f"total venue_keys:          {len(vk_stats):,}")
print(f"  exact 1 booktitle:       {int((vk_stats['n_bt'] == 1).sum()):,} ({(vk_stats['n_bt'] == 1).mean()*100:.1f}%)")
print(f"  2 to 5 booktitles:       {int(vk_stats['n_bt'].between(2, 5).sum()):,}")
print(f"  > 5 booktitles:          {int((vk_stats['n_bt'] > 5).sum()):,}")

print("\nTop 8 venue_keys with most distinct booktitles:")
display(vk_stats.head(8))

print("Top 5 spellings for the 5 messiest venue_keys:")
for vk in vk_stats.head(5).index:
    vc = tmp.loc[tmp["venue_key"] == vk, "bt_clean"].value_counts()
    top5 = ", ".join(f"{k!r} ({v:,})" for k, v in vc.head(5).items())
    print(f"  [{vk}] (total={vc.sum():,}, distinct={len(vc)}): {top5}")

# --- 4) Direction B: cleaned booktitle -> distinct venue_key ---
bt_stats = (
    tmp.groupby("bt_clean")
       .agg(rows=("paper_key", "count"),
            n_vk=("venue_key", "nunique"))
       .sort_values(["n_vk", "rows"], ascending=[False, False])
)
multi_vk = bt_stats[bt_stats["n_vk"] > 1]
print("\n=== Direction B: cleaned booktitles spanning multiple venue_keys ===")
print(f"cleaned booktitles mapping to >1 venue_key: {len(multi_vk):,} of {len(bt_stats):,} "
      f"({multi_vk['rows'].sum():,} rows)")
print("Top 10 by row count:")
display(multi_vk.sort_values("rows", ascending=False).head(10))
for bt in multi_vk.sort_values("rows", ascending=False).head(5).index:
    vc = tmp.loc[tmp["bt_clean"] == bt, "venue_key"].value_counts()
    top5 = ", ".join(f"{k!r} ({v:,})" for k, v in vc.head(5).items())
    print(f"  {bt!r} -> {top5}")

# --- 5) Workshop heuristic & parent-sharing check ---
WS_RE = re.compile(r"\bworkshops?\b|@", re.IGNORECASE)
ws_map = {b: bool(WS_RE.search(b)) for b in bt_map.values()}
tmp["is_ws"] = tmp["bt_clean"].map(ws_map)

vk_ws = tmp.groupby("venue_key")["is_ws"].agg(["count", "sum"])
vk_ws["share"] = vk_ws["sum"] / vk_ws["count"]
n_ws_only = int((vk_ws["share"] == 1.0).sum())
n_mixed = int(((vk_ws["share"] > 0.0) & (vk_ws["share"] < 1.0)).sum())
n_main_only = int((vk_ws["share"] == 0.0).sum())

print("\n=== Workshop heuristic (\\bworkshops?\\b or '@' in booktitle) ===")
print(f"rows matching workshop pattern: {int(tmp['is_ws'].sum()):,} of {n0:,} ({tmp['is_ws'].mean()*100:.2f}%)")
print(f"venue_keys with ONLY workshop rows:     {n_ws_only:,}")
print(f"venue_keys MIXED (main + workshop):     {n_mixed:,}")
print(f"venue_keys with NO workshop rows:       {n_main_only:,}")

print("\nRandom sample of 15 rows matching workshop pattern:")
display(tmp.loc[tmp["is_ws"], ["paper_key", "venue_key", "crossref", "bt_clean"]]
           .sample(min(15, int(tmp["is_ws"].sum())), random_state=42))

distinct raw booktitle:             8,547
distinct cleaned booktitle (NFC/ws):8,547
raw booktitle strings changed:      0
INFO - cleaned booktitles with control/format chars: 0
distinct venue_key:                 4,232 (changed by NFC/ws: 0)

=== Breakdown by key_prefix ===


,rows,unique_venue_key,unique_bt_clean
key_prefix,,,
conf,1897209,4230,8428
journals,2910,2,142
books,40,1,2



-- Non-conf prefix 'journals' (2,910 rows) top venue_keys:


,rows
venue_key,
corr,2523
entcs,387


Top cleaned booktitles in 'journals':


,rows
bt_clean,
QPL,171
GandALF,155
MFPS,150
TARK,132
ACT,108
ICLP Technical Communications,87
ICLR (Poster),85
ICLR (Workshop),73
LSFA,72


... (6 outputs left out)

Top 5 spellings for the 5 messiest venue_keys:
  [miccai] (total=10,586, distinct=151): 'MICCAI (1)' (874), 'MICCAI (2)' (864), 'MICCAI (3)' (840), 'MICCAI (4)' (599), 'MICCAI (6)' (518)
  [corr] (total=2,523, distinct=129): 'QPL' (171), 'GandALF' (155), 'TARK' (132), 'ACT' (108), 'ICLP Technical Communications' (87)
  [eccv] (total=8,320, distinct=114): 'ECCV Workshops (3)' (242), 'ECCV Workshops (2)' (240), 'ECCV Workshops (1)' (235), 'ECCV (5)' (216), 'ECCV (7)' (216)
  [icse] (total=6,145, distinct=98): 'ICSE' (1,590), 'ICSE (Companion Volume)' (901), 'ICSE Companion' (329), 'ICSE (2)' (215), 'ICSE (SEIP)' (195)
  [birthday] (total=1,779, distinct=89): 'Computability and Complexity' (42), 'Software, Services, and Systems' (38), 'Adventures Between Lower Bounds and Higher Altitudes' (36), 'The New Codebreakers' (33), 'From Software Engineering to Formal Methods and Tools, and Back' (32)

=== Direction B: cleaned booktitles spanning multiple venue_keys ===
cleaned booktitles mapping 

,rows,n_vk
bt_clean,,
ICC,11262,2
ICLR,10267,2
SMC,6904,2
WCNC,5652,2
ICCE,5403,2
ICIS,4959,4
ECC,4790,2
AAMAS,4232,2
ICIT,3880,4


  'ICC' -> 'icc' (11,065), 'iotdcc' (197)
  'ICLR' -> 'iclr' (10,241), 'corr' (26)
  'SMC' -> 'smc' (6,815), 'smc2' (89)
  'WCNC' -> 'wcnc' (5,646), 'wcnc2' (6)
  'ICCE' -> 'iccel' (2,868), 'icce' (2,535)

=== Workshop heuristic (\bworkshops?\b or '@' in booktitle) ===
rows matching workshop pattern: 103,815 of 1,900,159 (5.46%)
venue_keys with ONLY workshop rows:     447
venue_keys MIXED (main + workshop):     486
venue_keys with NO workshop rows:       3,299

Random sample of 15 rows matching workshop pattern:


,paper_key,venue_key,crossref,bt_clean
1607185,conf/er/CarneinAT17,er,conf/er/2017w,ER Workshops
389746,conf/miccai/NistelrooijV24,miccai,conf/miccai/2024toothfairy,ToothFairy/3DTeethLand/STS@MICCAI
404168,conf/ccs/JuutiAA19,ccs,conf/ccs/2019aisec,AISec@CCS
840177,conf/icc/ChengLH20,icc,conf/icc/2020w,ICC Workshops
1795548,conf/hipeac/MuttilloGFP20,hipeac,conf/hipeac/2020parma,PARMA-DITAM@HiPEAC
517795,conf/uai/AffeldtI15,uai,conf/uai/2015aci,ACI@UAI
458349,conf/eccv/KinliOK22,eccv,conf/eccv/2022-w2,ECCV Workshops (2)
1393707,conf/cvpr/Bernal21,cvpr,conf/cvpr/2021w,CVPR Workshops
570769,conf/ecir/KutuzovK16,ecir,conf/ecir/2016newsir,NewsIR@ECIR
691689,conf/pkdd/AlsaidiDLMMC21,pkdd,conf/pkdd/2021-w1,PKDD/ECML Workshops (1)


### T04b. Normalisasi nama conference/venue

**Aturan.**
1. **ID kanonik (`venue_key`):** Menggunakan segmen kedua dari `paper_key` (setelah normalisasi Unicode NFC dan perapian spasi) tanpa penggabungan lintas kunci. Pengukuran T04a menunjukkan bahwa `455` string `booktitle` bersih dipakai oleh lebih dari satu `venue_key` (`195.473` baris, misalnya akronim `ICC` untuk `icc` dan `iotdcc`), sehingga penggabungan berbasis teks `booktitle` akan menyatukan konferensi yang berbeda.
2. **Flag prefix (`in_conf_prefix`) dan workshop (`is_workshop`):** Tidak ada baris yang dibuang pada langkah ini. Kolom boolean `in_conf_prefix` menandai baris berawalan `conf/`, sedangkan `is_workshop` menandai baris yang `booktitle`-nya memuat kata `Workshop`/`Workshops` atau simbol `@` (`103.815` baris; presisi `15/15` pada sampel acak). Workshop terafiliasi konferensi besar sudah berbagi `venue_key` dengan konferensi induknya (misalnya `conf/cvpr/2021w` → `cvpr`), sedangkan workshop mandiri tetap memakai `venue_key` masing-masing.
3. **Nama tampilan 1-ke-1 (`venue_name`):** Setiap `venue_key` dipetakan ke satu nama tampilan berdasarkan `booktitle` bersih yang paling sering muncul pada baris non-workshop (atau seluruh baris jika `venue_key` tersebut hanya berisi workshop), setelah menghapus sufiks nomor volume/bagian di akhir seperti ` (1)`, ` (2)`, atau ` (Companion Volume)` agar volume utama seperti `MICCAI` dan `ECCV` tidak tertutup oleh label `ECCV Workshops (3)`.

**Keterbatasan.**
1. Kunci `journals/corr` (`2.523` baris, `in_conf_prefix = False`) merupakan pseudo-venue arXiv yang menampung `129` `booktitle` berbeda (seperti `QPL`, `GandALF`, dan `ICLR (Poster)`), bukan satu konferensi tunggal. Dampaknya diuji lewat analisis sensitivitas pada Task 2.
2. Kunci `conf/birthday` (`1.779` baris) adalah kategori volume penghormatan (*Festschrift*) di DBLP, bukan konferensi tahunan tunggal.

In [46]:
# T04b - Venue normalization: canonical venue_key, display venue_name, and is_workshop flag
import unicodedata

assert not any(s["step"].startswith("T04") for s in funnel), "T04 already applied; rerun T00-T03 first"
n0 = len(df)

def norm_str(s):
    """NFC + collapse all Unicode whitespace to single spaces + strip."""
    return "" if s is None else " ".join(unicodedata.normalize("NFC", str(s)).split())

# --- 1) Clean venue_key and booktitle strings (defensive NFC + whitespace) ---
vk_map = {v: norm_str(v) for v in df["venue_key"].unique()}
df["venue_key"] = df["venue_key"].map(vk_map)

bt_map = {b: norm_str(b) for b in df["booktitle"].unique()}
bt_clean = df["booktitle"].map(bt_map)

# --- 2) Boolean flags: in_conf_prefix and is_workshop ---
df["in_conf_prefix"] = df["key_prefix"].eq("conf")
WS_RE = re.compile(r"\bworkshops?\b|@", re.IGNORECASE)
ws_map = {b: bool(WS_RE.search(b)) for b in bt_map.values()}
df["is_workshop"] = bt_clean.map(ws_map)

# --- 3) Derive 1-to-1 display venue_name per venue_key ---
# Prefer non-workshop rows when available; strip trailing volume/part/companion qualifiers for voting
PART_SUFFIX_RE = re.compile(r"\s*\((?:\d+|Companion(?:\s+Volume)?)\)\s*$", re.IGNORECASE)

def clean_display_candidate(bt):
    stripped = PART_SUFFIX_RE.sub("", bt).strip()
    return stripped if stripped else bt

cand_bt = bt_clean.map({b: clean_display_candidate(b) for b in bt_map.values()})

has_main = df.groupby("venue_key")["is_workshop"].transform(lambda s: (~s).any())
use_for_name = (~df["is_workshop"]) | (~has_main)

name_counts = (
    pd.DataFrame({"venue_key": df.loc[use_for_name, "venue_key"], "cand": cand_bt[use_for_name]})
      .value_counts()
      .reset_index(name="n")
)
name_counts["len"] = name_counts["cand"].str.len()
name_counts = name_counts.sort_values(["venue_key", "n", "len", "cand"], ascending=[True, False, True, True])
venue_name_map = name_counts.drop_duplicates("venue_key").set_index("venue_key")["cand"].to_dict()

df["venue_name"] = df["venue_key"].map(venue_name_map)

# --- 4) Evidence & verification ---
vk_prefixes = df.groupby("venue_key")["key_prefix"].nunique()
shared_vk = vk_prefixes[vk_prefixes > 1].index.tolist()
print(f"venue_keys spanning multiple key_prefix values: {shared_vk}")
if shared_vk:
    display(df[df["venue_key"].isin(shared_vk)].groupby(["venue_key", "key_prefix"]).size().rename("rows").to_frame())

print("\nResolved venue_name for the 5 messiest venue_keys from T04a:")
messy_keys = ["miccai", "corr", "eccv", "icse", "birthday"]
messy_tbl = (
    df[df["venue_key"].isin(messy_keys)]
      .groupby(["venue_key", "venue_name", "in_conf_prefix"], as_index=False)
      .agg(rows=("paper_key", "count"),
           workshop_rows=("is_workshop", "sum"))
)
messy_tbl["workshop_%"] = (messy_tbl["workshop_rows"] / messy_tbl["rows"] * 100).round(1)
display(messy_tbl.sort_values("rows", ascending=False))

print("\nTop 10 venue_keys by publication count after T04:")
top10_tbl = (
    df.groupby(["venue_key", "venue_name", "in_conf_prefix"], as_index=False)
      .agg(rows=("paper_key", "count"),
           workshop_rows=("is_workshop", "sum"))
      .sort_values("rows", ascending=False)
      .head(10)
)
top10_tbl["workshop_%"] = (top10_tbl["workshop_rows"] / top10_tbl["rows"] * 100).round(1)
display(top10_tbl)

# --- 5) Log step and run PASS/FAIL checks ---
log_step(
    "T04 Venue normalization (canonical venue_key, display venue_name, is_workshop flag)",
    n0, len(df),
    f"unique venue_key: {df['venue_key'].nunique():,}; workshop rows: {int(df['is_workshop'].sum()):,}"
)

checks = {
    "no rows removed":                           len(df) == n0,
    "venue_key has no nulls or blanks":          bool(df["venue_key"].fillna("").str.strip().ne("").all()),
    "venue_name has no nulls or blanks":         bool(df["venue_name"].fillna("").str.strip().ne("").all()),
    "1-to-1 mapping venue_key -> venue_name":    int(df.groupby("venue_key")["venue_name"].nunique().max()) == 1,
    "is_workshop and in_conf_prefix are bool":   df["is_workshop"].dtype == "bool" and df["in_conf_prefix"].dtype == "bool",
    "venue_key normalization is idempotent":     all(norm_str(x) == x for x in df["venue_key"].unique()),
    "paper_key still unique":                    bool(df["paper_key"].is_unique),
}
for name, ok in checks.items():
    print("PASS" if ok else "FAIL", "-", name)
display(pd.DataFrame(funnel))

venue_keys spanning multiple key_prefix values: ['sp']


rows
venue_key key_prefix      
sp        books         40
          conf        1855


Resolved venue_name for the 5 messiest venue_keys from T04a:


,venue_key,venue_name,in_conf_prefix,rows,workshop_rows,workshop_%
4,miccai,MICCAI,True,10586,4555,43.0
2,eccv,ECCV,True,8320,1740,20.9
3,icse,ICSE,True,6145,1947,31.7
1,corr,QPL,False,2523,724,28.7
0,birthday,Principles of Verification,True,1779,0,0.0



Top 10 venue_keys by publication count after T04:


,venue_key,venue_name,in_conf_prefix,rows,workshop_rows,workshop_%
956,cvpr,CVPR,True,22780,5345,23.5
1786,icassp,ICASSP,True,21608,481,2.2
2312,igarss,IGARSS,True,19688,0,0.0
3144,nips,NeurIPS,True,19632,180,0.9
1626,hci,HCI,True,19370,0,0.0
34,aaai,AAAI,True,19277,844,4.4
669,chi,CHI,True,14866,188,1.3
1262,embc,EMBC,True,14679,0,0.0
1818,icc,ICC,True,14330,3265,22.8
71,acl,ACL,True,12788,195,1.5


PASS - no rows removed
PASS - venue_key has no nulls or blanks
PASS - venue_name has no nulls or blanks
PASS - 1-to-1 mapping venue_key -> venue_name
PASS - is_workshop and in_conf_prefix are bool
PASS - venue_key normalization is idempotent
PASS - paper_key still unique


,step,rows_before,rows_after,removed,note
0,"Raw extract: inproceedings, 2015-2025 (done at...",1901774,1901774,0,
1,T01 Drop rows missing author/booktitle/venue_k...,1901774,1900159,1615,blank titles kept
2,"T02 Year: cast year_raw to int16, consistency ...",1900159,1900159,0,rows disagreeing on >=1 signal: 7006; on both:...
3,"T03 Author normalization (NFC, whitespace, dro...",1900159,1900159,0,blank entries removed: 0
4,"T04 Venue normalization (canonical venue_key, ...",1900159,1900159,0,"unique venue_key: 4,232; workshop rows: 103,815"


### T05. Deteksi dan penghapusan data duplikat

**Aturan.** Pemeriksaan duplikat dijalankan setelah normalisasi author (T03) dan venue (T04) agar perbandingan dilakukan pada nilai kanonik, dalam dua tingkat:
1. **Tingkat 1 (`paper_key`):** Memastikan tidak ada kunci record ganda (diharapkan `0` duplikat).
2. **Tingkat 2 (duplikat metadata pada venue dan tahun yang sama):** Judul dinormalisasi sementara (*Unicode NFC*, *casefold*, dan penghapusan tanda baca/spasi berlebih) khusus untuk pencocokan, lalu dicocokkan berdasarkan `(venue_key, year, norm_title)`. Kandidat hanya dianggap duplikat dan dihapus jika **himpunan `author_id` juga identik**. Judul kosong, judul sangat pendek (`< 8` karakter), serta judul generik yang sah berulang pada workshop/sesi berbeda di venue yang sama (seperti *Preface*, *Front Matter*, *Keynote*, *Tutorial*) dikecualikan dari pencocokan.

**Aturan pemilihan baris yang dipertahankan.** Dalam setiap kelompok duplikat, baris dengan `paper_key` paling awal secara leksikografis dipertahankan (`kept_key`), sedangkan sisanya dicatat ke `df_duplicates` dan `df_rejected`.

In [47]:
# T05 - Duplicate detection and removal (Tier 1 paper_key, Tier 2 metadata + author set)
assert not any(s["step"].startswith("T05") for s in funnel), "T05 already applied; rerun from T00 first"
n0 = len(df)

# --- 1) Tier 1: exact paper_key uniqueness ---
assert df["paper_key"].is_unique, f"Tier 1 failed: {int(df['paper_key'].duplicated().sum())} duplicate paper_keys"
print("Tier 1 (paper_key duplicates): 0")

# --- 2) Tier 2: temporary normalized title for matching only ---
GENERIC_TITLES = {
    "preface", "foreword", "front matter", "table of contents", "author index",
    "welcome", "committee", "keynote", "keynote talk", "invited talk",
    "panel", "tutorial", "posters", "demos", "session overview"
}

norm_title = (
    df["title"].fillna("")
      .str.normalize("NFC")
      .str.casefold()
      .str.replace(r"[\W_]+", " ", regex=True)
      .str.strip()
)
matchable = norm_title.str.len().ge(8) & ~norm_title.isin(GENERIC_TITLES)

# Narrow down to rows sharing (venue_key, year, norm_title) before comparing author sets
cand_mask = matchable & pd.DataFrame({
    "venue_key": df["venue_key"],
    "year": df["year"],
    "norm_title": norm_title,
}).duplicated(keep=False)

cand = (
    df.loc[cand_mask, ["paper_key", "venue_key", "year", "title", "author_id"]]
      .assign(_norm_title=norm_title[cand_mask],
              _author_set=df.loc[cand_mask, "author_id"].map(frozenset))
      .sort_values("paper_key", ascending=True)
)

kept_series = cand.groupby(["venue_key", "year", "_norm_title", "_author_set"])["paper_key"].transform("first")
dup_cand = cand[cand["paper_key"] != kept_series].copy()
dup_cand["kept_key"] = kept_series[dup_cand.index]

# --- 3) Evidence: candidate counts, confirmed duplicates, and examples ---
n_cand_groups = cand.groupby(["venue_key", "year", "_norm_title"]).ngroups if len(cand) else 0
n_diff_author = len(cand) - (len(dup_cand) + dup_cand["kept_key"].nunique()) if len(dup_cand) else len(cand)

print(f"Tier 2 candidate rows (same venue_key + year + norm_title): {len(cand):,} across {n_cand_groups:,} title groups")
print(f"  Confirmed duplicates removed (author set also matches):    {len(dup_cand):,}")
print(f"  Candidate rows kept (different author sets):               {n_diff_author:,}")

if len(dup_cand):
    print("\nSample of removed duplicates (up to 8):")
    display(dup_cand[["paper_key", "kept_key", "venue_key", "year", "title"]].head(8))

# --- 4) Remove confirmed duplicates and record in audit logs ---
df_duplicates = dup_cand[["paper_key", "kept_key", "venue_key", "year", "title"]].reset_index(drop=True)

if len(dup_cand):
    dup_idx = dup_cand.index
    reason = "duplicate of " + dup_cand["kept_key"]
    reject(df.loc[dup_idx], "T05 duplicates", reason)
    df = df.drop(index=dup_idx).copy()

log_step(
    "T05 Remove Tier-2 duplicates (same venue_key, year, normalized title, and author set)",
    n0, len(df),
    f"confirmed duplicates removed: {len(df_duplicates):,}"
)

# --- 5) PASS/FAIL checks ---
checks = {
    "rows kept + duplicates removed == rows before": len(df) + len(df_duplicates) == n0,
    "df_duplicates paper_key is unique":             bool(df_duplicates["paper_key"].is_unique),
    "no removed duplicate remains in df":            not df["paper_key"].isin(df_duplicates["paper_key"]).any(),
    "all kept_keys still exist in df":               bool(df_duplicates["kept_key"].isin(df["paper_key"]).all()),
    "df_rejected paper_key is unique":               bool(df_rejected["paper_key"].is_unique),
    "paper_key still unique in df":                  bool(df["paper_key"].is_unique),
}
for name, ok in checks.items():
    print("PASS" if ok else "FAIL", "-", name)
display(pd.DataFrame(funnel))

Tier 1 (paper_key duplicates): 0
Tier 2 candidate rows (same venue_key + year + norm_title): 2,714 across 1,295 title groups
  Confirmed duplicates removed (author set also matches):    1,121
  Candidate rows kept (different author sets):               493

Sample of removed duplicates (up to 8):


,paper_key,kept_key,venue_key,year,title
910529,conf/3dic/LahijiLS16a,conf/3dic/LahijiLS16,3dic,2016,3D integration and challenges for advanced RF ...
1614508,conf/ACISicis/HuX17a,conf/ACISicis/HuX17,ACISicis,2017,A machinima system based on network game.
1614086,conf/ACISicis/WangCZZ17a,conf/ACISicis/WangCZZ17,ACISicis,2017,Real-time image detail enhancement implementin...
1614159,conf/ACISicis/WangL17a,conf/ACISicis/WangL17,ACISicis,2017,A parallel image processing platform based on ...
918216,conf/ACMdis/KimJKKP19a,conf/ACMdis/KimJKKP19,ACMdis,2019,muRedder: Shredding Speaker for Ephemeral Musi...
940623,conf/IEEEicci/ShaukatYAMM19a,conf/IEEEicci/ShaukatYAMM19,IEEEicci,2019,Towards Automatic Recognition of Sounds Observ...
49805,conf/IEEEssd/SeddikB22a,conf/IEEEssd/SeddikB22,IEEEssd,2022,Synthesis and high-performance analysis of Syn...
130128,conf/aaai/BulgarovN18a,conf/aaai/BulgarovN18,aaai,2018,Proposition Entailment in Educational Applicat...


PASS - rows kept + duplicates removed == rows before
PASS - df_duplicates paper_key is unique
PASS - no removed duplicate remains in df
PASS - all kept_keys still exist in df
PASS - df_rejected paper_key is unique
PASS - paper_key still unique in df


,step,rows_before,rows_after,removed,note
0,"Raw extract: inproceedings, 2015-2025 (done at...",1901774,1901774,0,
1,T01 Drop rows missing author/booktitle/venue_k...,1901774,1900159,1615,blank titles kept
2,"T02 Year: cast year_raw to int16, consistency ...",1900159,1900159,0,rows disagreeing on >=1 signal: 7006; on both:...
3,"T03 Author normalization (NFC, whitespace, dro...",1900159,1900159,0,blank entries removed: 0
4,"T04 Venue normalization (canonical venue_key, ...",1900159,1900159,0,"unique venue_key: 4,232; workshop rows: 103,815"
5,"T05 Remove Tier-2 duplicates (same venue_key, ...",1900159,1899038,1121,"confirmed duplicates removed: 1,121"


### T06. Verifikasi integritas akhir, statistik ringkasan (Task 1.4), dan ekspor data bersih

**Tujuan.** Sebelum mengekspor dataset bersih untuk digunakan pada Task 2 dan Task 3, sel ini menjalankan tiga tahap penutup Task 1:
1. **Pemeriksaan integritas menyeluruh (`PASS/FAIL`):** Memastikan konservasi jumlah baris terhadap data mentah (`df + df_rejected == raw`), keunikan `paper_key`, kelengkapan `author_id` dan `year`, pemetaan 1-ke-1 `venue_key -> venue_name`, serta tidak tersisanya duplikat Tingkat 2.
2. **Statistik deskriptif sebelum vs sesudah preprocessing (Task 1.4):** Membandingkan jumlah publikasi, total entri dan jumlah penulis unik, rata-rata/median penulis per publikasi, jumlah venue unik, serta distribusi tahun (2015–2025).
3. **Ekspor artefak bersih:** Menyimpan `data/clean_inproceedings.parquet` (dengan kolom mentah dan kolom audit sementara dibuang agar skema bersih), `data/rejected_rows.parquet` untuk audit, serta `data/clean_summary.json` berisi ringkasan dan SHA-256 berkas untuk verifikasi lintas anggota tim.

In [48]:
# T06 - Final integrity checks, Task 1.4 summary statistics, and Parquet/JSON export

# --- 1) Final end-to-end integrity assertions ---
norm_title_final = (
    df["title"].fillna("")
      .str.normalize("NFC")
      .str.casefold()
      .str.replace(r"[\W_]+", " ", regex=True)
      .str.strip()
)
GENERIC_TITLES = {
    "preface", "foreword", "front matter", "table of contents", "author index",
    "welcome", "committee", "keynote", "keynote talk", "invited talk",
    "panel", "tutorial", "posters", "demos", "session overview"
}
matchable_final = norm_title_final.str.len().ge(8) & ~norm_title_final.isin(GENERIC_TITLES)
cand_final_mask = matchable_final & pd.DataFrame({
    "venue_key": df["venue_key"],
    "year": df["year"],
    "norm_title": norm_title_final,
}).duplicated(keep=False)

if cand_final_mask.any():
    rem_dups = pd.DataFrame({
        "venue_key": df.loc[cand_final_mask, "venue_key"],
        "year": df.loc[cand_final_mask, "year"],
        "norm_title": norm_title_final[cand_final_mask],
        "author_set": df.loc[cand_final_mask, "author_id"].map(frozenset),
    }).duplicated().sum()
else:
    rem_dups = 0

ids_final = set(a for L in df["author_id"] for a in L)

final_checks = {
    "row conservation (len(df) + len(df_rejected) == len(raw))": len(df) + len(df_rejected) == len(raw),
    "paper_key has no nulls and is unique":                     bool(df["paper_key"].notna().all() and df["paper_key"].is_unique),
    "author_id non-empty, no blanks, no intra-paper dupes":      bool(
        df["author_id"].map(len).ge(1).all()
        and ("" not in ids_final)
        and df["author_id"].map(lambda x: len(set(x)) == len(x)).all()
    ),
    "n_authors matches len(author_id)":                          bool((df["n_authors"] == df["author_id"].map(len)).all()),
    "year is int16, no nulls, within 2015-2025":                 bool(df["year"].dtype == "int16" and df["year"].notna().all() and df["year"].between(YEAR_MIN, YEAR_MAX).all()),
    "venue_key and venue_name non-blank & strictly 1-to-1":      bool(
        df["venue_key"].fillna("").str.strip().ne("").all()
        and df["venue_name"].fillna("").str.strip().ne("").all()
        and int(df.groupby("venue_key")["venue_name"].nunique().max()) == 1
    ),
    "in_conf_prefix and is_workshop are bool with no nulls":     bool(
        df["in_conf_prefix"].dtype == "bool" and df["is_workshop"].dtype == "bool"
        and df["in_conf_prefix"].notna().all() and df["is_workshop"].notna().all()
    ),
    "zero Tier-2 duplicates remain":                             int(rem_dups) == 0,
}

print("=== Final Integrity Checks ===")
for name, ok in final_checks.items():
    print("PASS" if ok else "FAIL", "-", name)
assert all(final_checks.values()), "One or more final integrity checks failed!"

# --- 2) Summary statistics before vs after preprocessing (Task 1.4) ---
raw_author_entries = int(raw["authors"].map(len).sum())
clean_author_entries = int(df["n_authors"].sum())
raw_unique_authors = len(set(a for L in raw["authors"] for a in L))
clean_unique_authors = len(ids_final)

summary_stats = pd.DataFrame([
    ("Jumlah publikasi (paper_key)",             f"{len(raw):,}",                        f"{len(df):,}",                        f"{len(df) - len(raw):+,}"),
    ("Total entri author-paper",                 f"{raw_author_entries:,}",              f"{clean_author_entries:,}",            f"{clean_author_entries - raw_author_entries:+,}"),
    ("Jumlah penulis unik (author_id)",          f"{raw_unique_authors:,}",              f"{clean_unique_authors:,}",            f"{clean_unique_authors - raw_unique_authors:+,}"),
    ("Rata-rata penulis per publikasi",          f"{raw_author_entries / len(raw):.2f}", f"{df['n_authors'].mean():.2f}",       ""),
    ("Median (IQR) penulis per publikasi",       f"{raw['authors'].map(len).median():.0f}", f"{df['n_authors'].median():.0f} ({df['n_authors'].quantile(0.25):.0f}-{df['n_authors'].quantile(0.75):.0f})", ""),
    ("Jumlah venue unik (venue_key, semua)",     f"{raw['venue_key'].nunique():,}",      f"{df['venue_key'].nunique():,}",      f"{df['venue_key'].nunique() - raw['venue_key'].nunique():+,}"),
    ("Jumlah venue unik (prefix conf/)",         f"{raw.loc[raw['key_prefix']=='conf', 'venue_key'].nunique():,}",
                                                 f"{df.loc[df['in_conf_prefix'], 'venue_key'].nunique():,}", ""),
    ("Publikasi workshop (is_workshop=True)",    "-",                                    f"{int(df['is_workshop'].sum()):,} ({df['is_workshop'].mean()*100:.2f}%)", ""),
], columns=["Metrik", "Sebelum Preprocessing (Raw)", "Sesudah Preprocessing (Clean)", "Selisih"])

print("\n=== Ringkasan Statistik Dataset (Task 1.4) ===")
with pd.option_context("display.max_colwidth", None):
    display(summary_stats)

print("\n=== Distribusi Publikasi per Tahun (2015-2025, Clean) ===")
yr_clean = df.groupby("year").agg(
    papers=("paper_key", "count"),
    workshop_papers=("is_workshop", "sum"),
    unique_venues=("venue_key", "nunique"),
)
yr_clean["share_%"] = (yr_clean["papers"] / len(df) * 100).round(2)
display(yr_clean)

# --- 3) Export clean parquet, rejected audit parquet, and summary JSON ---
EXPORT_COLS = [
    "paper_key", "author_id", "n_authors", "title", "year",
    "booktitle", "crossref", "key_prefix", "venue_key", "venue_name",
    "in_conf_prefix", "is_workshop",
]
CLEAN_SCHEMA = pa.schema([
    ("paper_key",      pa.string()),
    ("author_id",      pa.list_(pa.string())),
    ("n_authors",      pa.int16()),
    ("title",          pa.string()),
    ("year",           pa.int16()),
    ("booktitle",      pa.string()),
    ("crossref",       pa.string()),
    ("key_prefix",     pa.string()),
    ("venue_key",      pa.string()),
    ("venue_name",     pa.string()),
    ("in_conf_prefix", pa.bool_()),
    ("is_workshop",    pa.bool_()),
])

CLEAN_PARQUET    = OUT_DIR / "clean_inproceedings.parquet"
REJECTED_PARQUET = OUT_DIR / "rejected_rows.parquet"
CLEAN_JSON       = OUT_DIR / "clean_summary.json"

def write_parquet_atomic(table, path):
    tmp = path.with_name(path.name + ".tmp")
    pq.write_table(table, tmp, compression="snappy")
    tmp.replace(path)

df_export = df[EXPORT_COLS].reset_index(drop=True)
write_parquet_atomic(pa.Table.from_pandas(df_export, schema=CLEAN_SCHEMA, preserve_index=False), CLEAN_PARQUET)

rej_export = df_rejected.drop(columns=["year_vs_key", "year_vs_xref"], errors="ignore").reset_index(drop=True)
write_parquet_atomic(pa.Table.from_pandas(rej_export, preserve_index=False), REJECTED_PARQUET)

def file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while chunk := f.read(1024 * 1024):
            h.update(chunk)
    return h.hexdigest()

clean_sha = file_sha256(CLEAN_PARQUET)
clean_meta = {
    "source_raw_parquet": str(RAW_PARQUET),
    "clean_parquet": str(CLEAN_PARQUET),
    "clean_parquet_size_bytes": CLEAN_PARQUET.stat().st_size,
    "clean_parquet_sha256": clean_sha,
    "columns": EXPORT_COLS,
    "n_papers_raw": int(len(raw)),
    "n_papers_clean": int(len(df_export)),
    "n_papers_removed": int(len(df_rejected)),
    "n_unique_authors": int(clean_unique_authors),
    "n_total_author_entries": int(clean_author_entries),
    "n_unique_venues_all": int(df_export["venue_key"].nunique()),
    "n_unique_venues_conf_prefix": int(df_export.loc[df_export["in_conf_prefix"], "venue_key"].nunique()),
    "n_workshop_papers": int(df_export["is_workshop"].sum()),
    "papers_by_year": {int(k): int(v) for k, v in df_export["year"].value_counts().sort_index().items()},
    "funnel": funnel,
    "exported_at": time.strftime("%Y-%m-%d %H:%M:%S"),
}
tmp_json = CLEAN_JSON.with_name(CLEAN_JSON.name + ".tmp")
tmp_json.write_text(json.dumps(clean_meta, indent=2, ensure_ascii=False), encoding="utf-8")
tmp_json.replace(CLEAN_JSON)

# Update N_final in provenance table from A1/A5
prov.loc[prov["fakta"] == "N_final (setelah preprocessing)", ["nilai", "sumber"]] = [
    f"{len(df_export):,}", "Task 1 (T06)"
]

print(f"\nExported: {CLEAN_PARQUET} ({CLEAN_PARQUET.stat().st_size:,} bytes)")
print(f"SHA-256:  {clean_sha}")
print(f"Exported: {REJECTED_PARQUET} ({REJECTED_PARQUET.stat().st_size:,} bytes)")
print(f"Exported: {CLEAN_JSON} ({CLEAN_JSON.stat().st_size:,} bytes)")

=== Final Integrity Checks ===
PASS - row conservation (len(df) + len(df_rejected) == len(raw))
PASS - paper_key has no nulls and is unique
PASS - author_id non-empty, no blanks, no intra-paper dupes
PASS - n_authors matches len(author_id)
PASS - year is int16, no nulls, within 2015-2025
PASS - venue_key and venue_name non-blank & strictly 1-to-1
PASS - in_conf_prefix and is_workshop are bool with no nulls
PASS - zero Tier-2 duplicates remain

=== Ringkasan Statistik Dataset (Task 1.4) ===


,Metrik,Sebelum Preprocessing (Raw),Sesudah Preprocessing (Clean),Selisih
0,Jumlah publikasi (paper_key),"1,901,774","1,899,038","-2,736"
1,Total entri author-paper,"7,432,518","7,428,192","-4,326"
2,Jumlah penulis unik (author_id),"1,738,301","1,738,301",+0
3,Rata-rata penulis per publikasi,3.91,3.91,
4,Median (IQR) penulis per publikasi,4,4 (3-5),
5,"Jumlah venue unik (venue_key, semua)","4,232","4,232",+0
6,Jumlah venue unik (prefix conf/),"4,230","4,230",
7,Publikasi workshop (is_workshop=True),-,"103,683 (5.46%)",



=== Distribusi Publikasi per Tahun (2015-2025, Clean) ===


,papers,workshop_papers,unique_venues,share_%
year,,,,
2015,161268,8968,1975,8.49
2016,162107,9484,2018,8.54
2017,168096,9458,2066,8.85
2018,176560,10344,2157,9.30
2019,188716,10508,2241,9.94
2020,171809,9717,2038,9.05
2021,177973,8707,2199,9.37
2022,178232,9613,2269,9.39
2023,200399,10880,2325,10.55



Exported: data\clean_inproceedings.parquet (205,006,040 bytes)
SHA-256:  778066063eb72e3db573fc8a50669f3f1e9b2faa6903393d47812242227fcf94
Exported: data\rejected_rows.parquet (323,203 bytes)
Exported: data\clean_summary.json (2,535 bytes)
